In [ ]:
## Figure 1: Merapi Plot with Inset Zoom of DOAS Stations
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

# --- Define Esri Physical Basemap (NO LABELS) ---
class EsriTerrain(cimgt.GoogleTiles): # Inherits the coordinate math
    def _image_url(self, tile):
        x, y, z = tile
        # Pulls from Esri's public tile server instead of Google
        return f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Physical_Map/MapServer/tile/{z}/{y}/{x}'
    
class GoogleSatellite(cimgt.GoogleTiles):
    def _image_url(self, tile):
        x, y, z = tile
        return f'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
    
# --- Data Coordinates ---
vent = [110.446111, -7.541389]
stations = {
    "Kaliurang": [110.424971, -7.601118],
    "Babadan": [110.410614, -7.526085],
    "Jrakah": [110.421595, -7.497221]
}

fig = plt.figure(figsize=(14, 8))

# Initialize the basemap provider
terrain_basemap = EsriTerrain()

# --- Main Regional Map ---
# Note: Axis projection must match the basemap's projection
ax = plt.axes(projection=terrain_basemap.crs)

# Set extent (coordinates are still provided in lat/lon, so we specify crs=PlateCarree)
ax.set_extent([100, 120, -10.0, 0.0], crs=ccrs.PlateCarree())

# Add the terrain basemap at zoom level 8
ax.add_image(terrain_basemap, 8)

# Plot Merapi Location
ax.plot(vent[0], vent[1], 'r^', markersize=8, label='Mt. Merapi', transform=ccrs.PlateCarree())
ax.text(vent[0]-2.60, vent[1]+0.05, 'Mount Merapi', transform=ccrs.PlateCarree(), fontweight='bold', fontsize=12)

# Adding city labels
cities = {
    "Yogyakarta": [110.3695, -7.7970],
    "Semarang": [110.4225, -6.9666],
}

for city, loc in cities.items():
    # Plot a small black dot for the city
    ax.plot(loc[0], loc[1], 'ko', markersize=4, transform=ccrs.PlateCarree())
    
    # Add the text label slightly offset to the right
    ax.text(loc[0]-1.5, loc[1]-0.05, city, transform=ccrs.PlateCarree(), 
            fontsize=10, fontstyle='italic',
            # Adding a subtle white glow makes text readable against the terrain
            bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=0.5))

# --- Create Inset Plot ---
# Place the inset box manually: [left, bottom, width, height]
ax_ins = fig.add_axes([0.55, 0.20, 0.35, 0.35], projection=terrain_basemap.crs)

# Set zoomed extent for the inset
pad = 0.08
ax_ins.set_extent([vent[0]-pad, vent[0]+0.1, vent[1]-pad, vent[1]+pad], crs=ccrs.PlateCarree())

# Initialize it
sat_basemap = GoogleSatellite()

# Add the same terrain basemap, but at a higher zoom level
ax_ins.add_image(sat_basemap, 13)

# Plot Vent in Inset
ax_ins.plot(vent[0], vent[1], '^', color='black', markersize=14, transform=ccrs.PlateCarree())
ax_ins.text(vent[0]+0.01, vent[1]-0.005, 'Active Vent', 
            transform=ccrs.PlateCarree(), fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Plot DOAS Stations in Inset
colors = ['blue', 'green', 'red']
for i, (name, coords) in enumerate(stations.items()):
    ax_ins.plot(coords[0], coords[1], 's', color=colors[i], markersize=8, markeredgecolor='black', transform=ccrs.PlateCarree())
    ax_ins.text(coords[0]+0.01, coords[1], name, transform=ccrs.PlateCarree(), fontsize=10, 
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
    
    # Draw dashed lines from stations to vent to show line of sight
    ax_ins.plot([coords[0], vent[0]], [coords[1], vent[1]], 'k--', linewidth=1, alpha=0.6, transform=ccrs.PlateCarree())

# Draw connecting lines between main map and inset
mark_inset(ax, ax_ins, loc1=2, loc2=3, fc="none", ec="black", linewidth=1.5)

# Add border to inset
for spine in ax_ins.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.5)

# Add gridlines to the Main Map
gl = ax.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), 
                  color='black', alpha=0.4, linestyle='--', linewidth=0.5)
gl.top_labels = False   # Hide top labels
gl.right_labels = False # Hide right labels
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'size': 10, 'color': 'black'}
gl.ylabel_style = {'size': 10, 'color': 'black'}

# Add gridlines to the Inset Map
gl_ins = ax_ins.gridlines(draw_labels=True, crs=ccrs.PlateCarree(), 
                          color='black', alpha=0.3, linestyle=':')
gl_ins.top_labels = False
gl_ins.right_labels = False
gl_ins.xformatter = LONGITUDE_FORMATTER
gl_ins.yformatter = LATITUDE_FORMATTER
gl_ins.xlabel_style = {'size': 8}
gl_ins.ylabel_style = {'size': 8}

# Positioned at 95% width, 95% height of the main axes
ax.text(0.95, 0.95, 'N', transform=ax.transAxes, 
        ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.annotate('', xy=(0.95, 0.95), xytext=(0.95, 0.88), 
            xycoords='axes fraction', 
            arrowprops=dict(facecolor='black', width=2, headwidth=10))


# Define scale bar location (Bottom Left of Java)
scale_lon = 101
scale_lat = -9.25

# 100 km width in degrees at this specific latitude
scale_width_deg = 0.906 

# Draw the main line
ax.plot([scale_lon, scale_lon + scale_width_deg], [scale_lat, scale_lat], 
        color='black', linewidth=2, transform=ccrs.PlateCarree())

# Draw the end caps 
cap_height = 0.05
ax.plot([scale_lon, scale_lon], [scale_lat - cap_height, scale_lat + cap_height], 
        color='black', linewidth=1.5, transform=ccrs.PlateCarree())
ax.plot([scale_lon + scale_width_deg, scale_lon + scale_width_deg], 
        [scale_lat - cap_height, scale_lat + cap_height], 
        color='black', linewidth=1.5, transform=ccrs.PlateCarree())

# Add the text label
ax.text(scale_lon + (scale_width_deg / 2), scale_lat + 0.08, '100 km', 
        transform=ccrs.PlateCarree(), ha='center', va='bottom', 
        fontsize=10, fontweight='bold')

# ---------------------------------------------------------------------- INSERT PATH TO SAVE FIGURE HERE ------------------------------------------------------------------
#plt.savefig(r"C:\XX\XX\XX.png", dpi=300, bbox_inches='tight',transparent=True)

plt.show()

In [ ]:
## Script for Downloading TROPOMI Data
import os
import time
import requests
import shapely.geometry
import shapely.wkt
from geopy.distance import distance
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# ---------------------------------------------- USER CREDENTIALS FOR Copernicus Data Space Ecosystem (CDSE) OAuth 2.0 -----------------------------------------------------------
# Registration can be done here at https://documentation.dataspace.copernicus.eu/Registration.html
USERNAME = "XXX"
PASSWORD = "XXX" 

token_lock = threading.Lock()
token_data = {"value": None, "expiry": 0}

# Automatically refreshs token which otherwise expires after a certain time
def get_fresh_token():
    auth_server_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
    data = {"client_id": "cdse-public", "grant_type": "password", "username": USERNAME, "password": PASSWORD}
    response = requests.post(auth_server_url, data=data, timeout=30)
    response.raise_for_status()
    json_data = response.json()
    return json_data["access_token"], time.time() + json_data.get("expires_in", 3600) - 60

def get_token():
    """Auto-refresh token if expired"""
    with token_lock:
        if time.time() >= token_data["expiry"]:
            token_data["value"], token_data["expiry"] = get_fresh_token()
            print("🔄 Token refreshed")
        return token_data["value"]

# Initialize
get_token()
headers = {"Authorization": f"Bearer {get_token()}"}

## Defining parameters for query

# Volcano location - Merapi
volcano_lat = -7.54
volcano_lon = 110.446
radius_km = 500

# Function to create a circular polygon
def create_circle_polygon(lat, lon, radius_km, num_points=64):
    points = []
    for angle in range(0, 360, int(360 / num_points)):
        destination = distance(kilometers=radius_km).destination((lat, lon), bearing=angle)
        points.append((destination.longitude, destination.latitude))
    points.append(points[0])
    polygon = shapely.geometry.Polygon(points)
    return polygon

# Generate polygon and convert to WKT
circle_polygon = create_circle_polygon(volcano_lat, volcano_lon, radius_km)
wkt_polygon = shapely.wkt.dumps(circle_polygon, rounding_precision=6)

# Build the filter for SO2 L2 Product
odata_filter = (
    f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt_polygon}')"
    f" and ContentDate/Start ge 2024-01-01T00:00:00.000Z"
    f" and ContentDate/Start lt 2026-01-01T00:00:00.000Z"
    f" and Collection/Name eq 'SENTINEL-5P'"
    f" and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' "
    f"and att/OData.CSC.StringAttribute/Value eq 'L2__SO2___')"
    f" and contains(Name, 'OFFL')"
)

# Handle pagination to get ALL products
base_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
current_url = f"{base_url}?$filter={odata_filter}"

products = []
page = 1

print("Searching for products...")
while current_url:
    headers = {"Authorization": f"Bearer {get_token()}"}  # Refresh token for each page
    response = requests.get(current_url, headers=headers, timeout=120)
    response.raise_for_status()
    data = response.json()
    
    page_products = data.get('value', [])
    products.extend(page_products)
    
    print(f"  Page {page}: Found {len(page_products)} products (total: {len(products)})")
    
    # Get next page URL (this is the key to getting all results!)
    current_url = data.get('@odata.nextLink')
    page += 1
    
    time.sleep(0.5)  # Small delay between pages

# Sort products by name
products.sort(key=lambda x: x['Name'])

print(f"\n✓ Total products found: {len(products)}\n")
for p in products[:5]:
    print(f"  - {p['Name']}")
if len(products) > 5:
    print(f"  ... and {len(products) - 5} more\n")

## Downloading products
MAX_WORKERS = 4
CHUNK_SIZE = 1024 * 1024  # 1 MB chunks

# ---------------------------------------------------------------------- INSERT PATH FOR DOWNLOAD FOLDER HERE ------------------------------------------------------------------
DOWNLOAD_DIR = r'XXXXX'
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

def download_product(product):
    """Get fresh token for each download"""
    product_id = product['Id']
    product_name = product['Name']
    local_filename = os.path.join(DOWNLOAD_DIR, product_name)

    if os.path.exists(local_filename):
        return f"Skipping: {product_name} (already exists)"

    download_url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
    
    # Retry with fresh token if needed
    for attempt in range(3):
        try:
            session = requests.Session()
            session.headers.update({"Authorization": f"Bearer {get_token()}"})  # Fresh token
            
            with session.get(download_url, stream=True, timeout=120) as r:
                if r.status_code == 401:  # Token expired
                    get_fresh_token()  # Force refresh
                    session.close()
                    continue
                
                r.raise_for_status()
                
                with open(local_filename, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
            
            session.close()
            return f"Downloaded: {product_name}"

        except Exception as e:
            if attempt == 2:  # Last attempt
                return f"Error downloading {product_name}: {e}"
            time.sleep(2)
    
    return f"Failed: {product_name}"

# Execute downloads
print(f"Starting downloads with {MAX_WORKERS} parallel workers...\n")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(download_product, p) for p in products]
    
    for future in as_completed(futures):
        print(future.result())

print("\nProcessing complete.")

In [ ]:
## Checking for any date gaps in TROPOMI Data Downloaded
import os
import re
from datetime import datetime, timedelta

# -------------------------------------------------------------------------- INSERT DOWNLOAD FOLDER PATH HERE -----------------------------------------------------------------------
DOWNLOAD_DIR = r'XXXXX'
# Set start and end dates of interest
START_DATE = datetime(2024, 1, 1)
END_DATE = datetime(2025, 12, 31)

def find_missing_dates():
    # 1. Generate the set of all expected dates
    expected_dates = set()
    current = START_DATE
    while current <= END_DATE:
        expected_dates.add(current.date())
        current += timedelta(days=1)

    # 2. Scan the downloads folder
    if not os.path.exists(DOWNLOAD_DIR):
        print("Download directory not found.")
        return

    files = os.listdir(DOWNLOAD_DIR)
    found_dates = set()
    
    # Regex to match the date part of the filename: YYYYMMDD
    # Example: S5P_OFFL_L2__SO2____20230601T...
    date_pattern = re.compile(r"(\d{8})T")

    for f in files:
        match = date_pattern.search(f)
        if match:
            date_str = match.group(1)
            date_obj = datetime.strptime(date_str, "%Y%m%d").date()
            found_dates.add(date_obj)

    # 3. Compare sets
    missing_dates = sorted(list(expected_dates - found_dates))

    # --- REPORT ---
    print("-" * 30)
    print(f"REPORT FOR: {START_DATE.date()} to {END_DATE.date()}")
    print(f"Files found:   {len(files)}")
    print(f"Unique days:   {len(found_dates)}")
    print(f"Missing days:  {len(missing_dates)}")
    print("-" * 30)

    if missing_dates:
        print("\nMissing Dates:")
        for d in missing_dates:
            print(f" - {d}")
    else:
        print("\nPerfect! No gaps found in the date range.")

if __name__ == "__main__":
    find_missing_dates()

In [ ]:
# Combined Script to run Sensitivity Analysis for Tracks, DU, and Drop Distances (Fromm Raw Satellite Data to Pareto Values)) can be done using the sensitivity_analysis.py script
# It is much faster to use the seperate python file ran using a suitable kernel with the necessary packages (those imported below + packages used by Grandin et al.'s (2024) flux calculation tool
# in order to benefit from parallisation using multithreading

In [ ]:
## Figure 3 (Part 1): Plotting Pareto Frontier (2023)
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load your results
# ------------------------------------------------------------------------- INSERT PATH FOR CSV FILE HERE --------------------------------------------------------------------------
df = pd.read_csv(r"XXX.csv") # Use the same CSV file that was generated by the sensitivity_analysis.py script

# Drop any rows where the math failed (0 overlapping days, etc.)
df = df.dropna(subset=['Spearman_Corr', 'Mean_Uncertainty'])

# 2. Sort by Objective 1 (Correlation) in ascending order
# If two settings have the exact same Correlation, the one with lower uncertainty goes first
df_sorted = df.sort_values(by=['Spearman_Corr', 'Mean_Uncertainty'], ascending=[False, True])

# 3. Compute the Pareto Frontier
pareto_frontier = []
min_uncertainty_so_far = float('inf')

for index, row in df_sorted.iterrows():
    # Because we sorted by MAE, every subsequent row has a WORSE (or equal) MAE.
    # Therefore, for a row to be on the Pareto frontier, it MUST have a STRICTLY BETTER 
    # (lower) Mean Uncertainty than anything we've looked at so far.
    if row['Mean_Uncertainty'] < min_uncertainty_so_far:
        pareto_frontier.append(row)
        min_uncertainty_so_far = row['Mean_Uncertainty']

pareto_df = pd.DataFrame(pareto_frontier)

# 4. Visualization
plt.figure(figsize=(10, 6))

# Plot all tested combinations in gray
plt.scatter(df['Spearman_Corr'], df['Mean_Uncertainty'], color='lightgray', label='All Tested Parameters')

# Plot the Pareto optimal combinations in red
plt.scatter(pareto_df['Spearman_Corr'], pareto_df['Mean_Uncertainty'], color='red', s=60, label='Pareto Frontier', zorder=5)

# Connect the Pareto points with a line to show the curve
plt.plot(pareto_df['Spearman_Corr'], pareto_df['Mean_Uncertainty'], color='red', linestyle='--', zorder=4)

#  FLIP THE X-AXIS 
# This forces the highest correlation values to the left (closer to the origin)
plt.gca().invert_xaxis()

plt.xlabel('Correlation \n Better Accuracy $\\leftarrow$', fontsize=18)
plt.ylabel('Mean Absolute Uncertainty (tons) \n $\leftarrow$ Better Precision', fontsize=18)
plt.title('B. Sensitivity Analysis: Satellite-Derived vs Ground-Based SO2 Flux (Jan 2024 - Dec 2025)', fontsize=20)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400, transparent=True, bbox_inches='tight')
plt.show()

# 5. Create a table image

# Select and format columns
table_df = pareto_df[
    ['Tracks', 'DU', 'Drops', 'Spearman_Corr', 'Mean_Uncertainty']
].copy()

# Round values for cleaner display
table_df['Spearman_Corr'] = table_df['Spearman_Corr'].round(4)
table_df['Mean_Uncertainty'] = table_df['Mean_Uncertainty'].round(2)

# Create figure
fig, ax = plt.subplots(figsize=(12, 0.6 * len(table_df) + 1))
ax.axis('off')

# Create table
table = ax.table(cellText=table_df.values, colLabels=[
        'Tracks', '$CA_{min}$', '$r_{max}$ (km)',
        'Correlation',
        'Uncertainty (tons)',
        #'Filtered Days'
    ],
    loc='center',
    cellLoc='center'
)

# Styling
table.auto_set_font_size(False)
table.set_fontsize(16)
table.scale(1.5, 2.1) 

# Header styling + alternating row colors
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props()
        cell.set_facecolor('#f2f2f2')
    elif row % 2 == 0:
        cell.set_facecolor('#fafafa')

# Title
plt.title(
    "Pareto Optimal Parameter Sets",
    fontsize=14,
    weight='bold',
    pad=12
)

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400, transparent=True, bbox_inches='tight')
plt.show()


In [ ]:
## Figure 3 (Part 2): Plotting Pareto Frontier (2024-25)
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load your results
# ----------------------------------------------------------------------- INSERT PATH FOR CSV FILE HERE ------------------------------------------------------------------------
df = pd.read_csv(r"XXX.csv")
# Drop any rows where the math failed (0 overlapping days, etc.)
df = df.dropna(subset=['Spearman_Corr', 'Mean_Uncertainty'])

# 2. Sort by Objective 1 (Correlation) in ascending order
# If two settings have the exact same Correlation, the one with lower uncertainty goes first
df_sorted = df.sort_values(by=['Spearman_Corr', 'Mean_Uncertainty'], ascending=[False, True])

# 3. Compute the Pareto Frontier
pareto_frontier = []
min_uncertainty_so_far = float('inf')

for index, row in df_sorted.iterrows():
    # Because we sorted by MAE, every subsequent row has a WORSE (or equal) MAE.
    # Therefore, for a row to be on the Pareto frontier, it MUST have a STRICTLY BETTER 
    # (lower) Mean Uncertainty than anything we've looked at so far.
    if row['Mean_Uncertainty'] < min_uncertainty_so_far:
        pareto_frontier.append(row)
        min_uncertainty_so_far = row['Mean_Uncertainty']

pareto_df = pd.DataFrame(pareto_frontier)

# 4. Visualization
plt.figure(figsize=(10, 6))

# Plot all tested combinations in gray
plt.scatter(df['Spearman_Corr'], df['Mean_Uncertainty'], color='lightgray', label='All Tested Parameters')

# Plot the Pareto optimal combinations in red
plt.scatter(pareto_df['Spearman_Corr'], pareto_df['Mean_Uncertainty'], color='red', s=60, label='Pareto Frontier', zorder=5)

# Connect the Pareto points with a line to show the curve
plt.plot(pareto_df['Spearman_Corr'], pareto_df['Mean_Uncertainty'], color='red', linestyle='--', zorder=4)

#  FLIP THE X-AXIS 
# This forces the highest correlation values to the left (closer to the origin)
plt.gca().invert_xaxis()

plt.xlabel('Correlation \n Better Accuracy $\\leftarrow$', fontsize=18)
plt.ylabel('Mean Absolute Uncertainty (tons) \n $\leftarrow$ Better Precision', fontsize=18)
plt.title('B. Sensitivity Analysis: Satellite-Derived vs Ground-Based SO2 Flux (Jan 2024 - Dec 2025)', fontsize=20)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400, transparent=True, bbox_inches='tight')
plt.show()

# 5. Create a table image

# Select and format columns
table_df = pareto_df[
    ['Tracks', 'DU', 'Drops', 'Spearman_Corr', 'Mean_Uncertainty']
].copy()

# Round values for cleaner display
table_df['Spearman_Corr'] = table_df['Spearman_Corr'].round(4)
table_df['Mean_Uncertainty'] = table_df['Mean_Uncertainty'].round(2)

# Create figure
fig, ax = plt.subplots(figsize=(12, 0.6 * len(table_df) + 1))
ax.axis('off')

# Create table
table = ax.table(cellText=table_df.values, colLabels=[
        'Tracks', '$CA_{min}$', '$r_{max}$ (km)',
        'Correlation',
        'Uncertainty (tons)',
        #'Filtered Days'
    ],
    loc='center',
    cellLoc='center'
)

# Styling
table.auto_set_font_size(False)
table.set_fontsize(16)
table.scale(1.5, 2.1) 

# Header styling + alternating row colors
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props()
        cell.set_facecolor('#f2f2f2')
    elif row % 2 == 0:
        cell.set_facecolor('#fafafa')

# Title
plt.title(
    "Pareto Optimal Parameter Sets",
    fontsize=14,
    weight='bold',
    pad=12
)

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400, transparent=True, bbox_inches='tight')
plt.show()


In [ ]:
# Scraper for Visual Smoke Data
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime, timedelta

class SmokeScraper:
    def __init__(self):
        self.base_url = "https://magma.esdm.go.id/v1/gunung-api/laporan-harian/"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }

    def parse_visual_text(self, text):
        """Extracts environmental and smoke metrics."""
        result = {
            "visibility": "N/A", "smoke_present": "No", "smoke_height_m": 0,
            "smoke_intensity": "N/A", "weather": "N/A", "wind_direction": "N/A",
            "temp_min": None, "temp_max": None, "humidity_min": None,
            "humidity_max": None, "pressure_min": None, "pressure_max": None
        }
        if not text: return result

        clean_text = ' '.join(text.split())
        sentences = [s.strip() for s in re.split(r'\.(?!\d)', clean_text) if s.strip()]

        def extract_min_max(s):
            nums = re.findall(r"(\d+(?:\.\d+)?)", s)
            if len(nums) >= 2: return float(nums[0]), float(nums[1])
            elif len(nums) == 1:
                val = float(nums[0])
                return val, val
            return None, None

        for sentence in sentences:
            low_s = sentence.lower()
            if "terlihat" in low_s or "kabut" in low_s:
                if "terlihat jelas" in low_s and "kabut" in low_s: result["visibility"] = "Clear to Foggy"
                elif "tertutup kabut" in low_s: result["visibility"] = "Foggy"
                elif "terlihat jelas" in low_s: result["visibility"] = "Clear"
            if "asap" in low_s or "kawah" in low_s:
                if "nihil" in low_s or "tidak teramati" in low_s:
                    result["smoke_present"] = "No"
                else:
                    result["smoke_present"] = "Yes"
                    h_match = re.search(r"(\d+)\s*meter", sentence)
                    if h_match: result["smoke_height_m"] = int(h_match.group(1))
                    i_match = re.search(r"intensitas\s+([\w\s-]+?)\s+(?:sekitar|tinggi|dari|puncak)", sentence, re.IGNORECASE)
                    if i_match: result["smoke_intensity"] = i_match.group(1).strip()
            if "cuaca" in low_s:
                w_match = re.search(r"Cuaca\s+([\w\s-]+),\s+angin\s+([\w\s-]+)", sentence, re.IGNORECASE)
                if w_match:
                    result["weather"] = w_match.group(1).strip()
                    result["wind_direction"] = w_match.group(2).strip()
            if "suhu" in low_s:
                result["temp_min"], result["temp_max"] = extract_min_max(sentence)
            if "kelembaba" in low_s:
                result["humidity_min"], result["humidity_max"] = extract_min_max(sentence)
            if "tekanan" in low_s:
                result["pressure_min"], result["pressure_max"] = extract_min_max(sentence)
        return result

    def fetch_daily_data(self, target_date):
        url = f"{self.base_url}{target_date}"
        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            if response.status_code != 200: return []
            
            soup = BeautifulSoup(response.text, 'html.parser')
            rows = soup.find_all('tr')
            daily_records = []

            for row in rows:
                cols = row.find_all('td')
                if len(cols) < 4: continue
                
                volcano_name = cols[1].get_text(strip=True)
                if "Merapi" in volcano_name:
                    # Initialize the single record for the date
                    record = {"date": target_date, "volcano_name": "Gunung Merapi"}
                    
                    # Wide Visual Data
                    visual_raw = cols[2].get_text(separator=" ", strip=True)
                    record.update(self.parse_visual_text(visual_raw))
                    
                    daily_records.append(record)
                            
            return daily_records
        except Exception as e:
            print(f"Error on {target_date}: {e}")
            return []

    def scrape_date_range(self, start_date_str, end_date_str):
        current_date = datetime.strptime(start_date_str, "%Y-%m-%d")
        end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
        all_data = []
        
        while current_date <= end_date:
            date_str = current_date.strftime("%Y-%m-%d")
            print(f"Scraping Smoke Data: {date_str}...")
            all_data.extend(self.fetch_daily_data(date_str))
            current_date += timedelta(days=1)
            time.sleep(1.2) 

        return pd.DataFrame(all_data)

# --- Execution ---
if __name__ == "__main__":
    scraper = SmokeScraper()
    #------------------------------------------------------------------------ Define date range ----------------------------------------------------------------------------------
    START, END = "2024-01-01", "2024-01-31"
    df_smoke = scraper.scrape_date_range(START, END)
    
    if not df_smoke.empty:
        filename = f"merapi_smoke_report_{START}_to_{END}.csv"
        df_smoke.to_csv(filename, index=False)
        print(f"\nSuccess! Smoke report saved: {filename}")
        print(df_smoke.head())

In [ ]:
# Scraper for GVP Weekly Reports Part 1: Scrap all raw paragraphs from the full page
from bs4 import BeautifulSoup
import os

# 1. Load HTML file. HTML file can be downloaded directly from the GVP website.
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/GVP Scaper")  # Navigate to folder with the data files
path = os.path.join(data_dir, "Global Volcanism Program _ Merapi.html")
with open(path, 'r', encoding='utf-8') as file:
    soup = BeautifulSoup(file, 'html.parser')

# 2. Locate the Weekly Reports section by finding its index header
index_header = soup.find(lambda tag: tag.name == "h5" and "Weekly Reports - Index" in tag.text)
weekly_tab = index_header.find_parent("div", class_="tabbed-content")

extracted_reports = []

# 3. Iterate through all the report headers (h5 tags)
for header in weekly_tab.find_all('h5'):
    # Skip the index header itself
    if "Index" in header.text:
        continue
    
    # Extract the date range and remove the citation button text
    date_range = header.get_text(strip=True).replace('Cite this Report', '').strip()
    
    # 4. Find the immediate sibling <p> tag which holds the actual report text
    report_paragraph = header.find_next_sibling('p')
    
    if report_paragraph:
        report_text = report_paragraph.get_text(strip=True)
        
        extracted_reports.append({
            'date': date_range,
            'text': report_text
        })

print(f"Successfully scraped {len(extracted_reports)} weekly reports.")

In [ ]:
# Scraper for GVP Weekly Reports Part 2: Filter by date range
from datetime import datetime

#----------------------------------------------------------------------- Define the date range -------------------------------------------------------------------------------------
target_start_date = datetime(2024, 1, 1)  
target_end_date = datetime(2025, 12, 31) 

filtered_reports = []

for entry in extracted_reports:
    raw_date_string = entry['date']
    
    try:
        # GVP dates usually look like "5 February-11 February 2026" or "27 December 2023-2 January 2024"
        # We split by the hyphen and grab the second part (the end date of the week)
        end_date_str = raw_date_string.split('-')[-1].strip()
        
        # Convert the string (e.g., "11 February 2026") into a datetime object
        report_date = datetime.strptime(end_date_str, "%d %B %Y")
        
        # Check if the report falls within your target window
        if target_start_date <= report_date <= target_end_date:
            filtered_reports.append(entry)
            
    except ValueError:
        # If a date is formatted weirdly and fails to parse, you can catch it here
        print(f"Could not parse date: {raw_date_string}")

print(f"Found {len(filtered_reports)} reports in the specified date range.")

In [ ]:
# Scraper for GVP Weekly Reports Part 3: Using Gemini API to extract structured metrics from the raw paragraphs
# ----------------------------------- Prior to running this script, ensure you have a .env file in the same directory with the following content: ------------------------------------
# GEMINI_API_KEY = your_api_key_here

import os
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from dotenv import load_dotenv


# Load the environment variables from the .env file which stores the GEMINI_API_KEY
load_dotenv() 

# Initialize the Gemini client 
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# 1. Define the exact data structure for a SINGLE report
class MerapiMetrics(BaseModel):
    date: str = Field(description="Exact date range of the report")
    pyroclastic_flow_count: int | None = Field(description="Total number of pyroclastic flows mentioned. If none are mentioned return 0.")
    pyroclastic_flow_distance_km: float | None = Field(description="Maximum runout distance of pyroclastic flows in kilometers.")
    avalanche_count: int | None = Field(description="Total lava avalanche events mentioned.")
    avalanche_distance_km: float | None = Field(description="Maximum avalanche runout distance in kilometers.")
    ashfall_occurrence: bool | None = Field(description="True if ashfall occurred during the week.")

# 2. Define the schema for the BATCH response
class BatchExtraction(BaseModel):
    extracted_data: list[MerapiMetrics]

all_structured_results = []
chunk_size = 15

# 3. Loop through the filtered reports in batches to save API limits
# Note: Ensure your 'filtered_reports' list from BeautifulSoup is loaded in your environment
for i in range(0, len(filtered_reports), chunk_size):
    chunk = filtered_reports[i : i + chunk_size]
    
    # Defining the prompt
    prompt = """
    You are extracting structured metrics from Merapi volcano weekly reports.

    Rules:
    - Extract ONLY information explicitly stated in the text.
    - If a metric is not mentioned, return null.
    - Distances must be in kilometers.
    - Counts must be integers representing the total number of events mentioned.
    - Ashfall occurrence should be true if ashfall is reported at least once during the week.

    For each report return one object with the following fields:
    date
    pyroclastic_flow_count
    pyroclastic_flow_distance_km
    avalanche_count
    avalanche_distance_km
    ashfall_occurrence

    Reports:
    """
    for entry in chunk:
        prompt += f"Date: {entry['date']}\nText: {entry['text']}\n---\n"
        
    print(f"Sending batch {i//chunk_size + 1} of {(len(filtered_reports)-1)//chunk_size + 1} to the API...")
    
    # Call the API, passing in our batch Pydantic schema
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=BatchExtraction,
            temperature=0.0, # Zero temperature is best for strict factual extraction
        ),
    )
    
    # Parse the JSON response
    batch_json = json.loads(response.text)
    
    # Add the extracted list to our master list
    all_structured_results.extend(batch_json['extracted_data'])
    
    # Brief pause between batches just to be completely safe on limits
    time.sleep(2)

# Load the structured list of dictionaries into pandas and view it
df = pd.DataFrame(all_structured_results)
print("\nExtraction Complete!")
print(df.head())

#----------------------------------------------------------------------- Define path for output CSV -------------------------------------------------------------------------------------
df.to_csv(r"XXXX\merapi_weekly_metrics_24_25.csv", index=False)

In [ ]:
## Figure 4: Combined Plots of SO2 + Thermal + Seismic + Volcanic Activity + Smoke (2023)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches  
from scipy.stats import norm

# ==============================================================================
# 1. FILE PATHS & CONFIGURATION
# ==============================================================================
# ------------------------------------------------------------------------Add Local Files Paths here --------------------------------------------------------------------------------
# Note that the output_0.8.csv (GROUND_TON_FILE) and bpptkg_seismicity_counts.csv (SEISMIC_FILE) files are not provided in the "Figure 4" folder since those are data provided by BPPTKG.
# These data are available from BPPTKG upon request. However, sample files are included to indicate how the format of these files would look like.

script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Figure 4")  # Navigate to folder with the data files

GROUND_TON_FILE = os.path.join(data_dir, "output_0.8.csv") # Provided by BPPTKG
SAT_FLUX_FILE = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-12-31_tracks-18_du-0.5_qa-0_sza-70_drops-none_P550.csv") # Generated from Post-Inversion calculation (Phase 2 in sensitivty_analysis.py)
CLOUD_FILE = os.path.join(data_dir, "TROPOMI_CloudFraction_mean_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-18_qa-0_sza-70_2023-06-01_2023-12-31.csv") # Generated from Mass Integration of TROPOMI SO2 L2 Product (Phase 1 in sensitivty_analysis.py)
SEISMIC_FILE = os.path.join(data_dir, "bpptkg_seismicity_counts.csv") # Provided by BPPTKG
THERMAL_FILE = os.path.join(data_dir, "fire_archive_J1V-C2_733161.csv") # Downloaded from NASA FIRMS
VOLCANIC_FILE = os.path.join(data_dir, "merapi_weekly_metrics_23.csv") # Scraped
SMOKE_FILE = os.path.join(data_dir, "merapi_smoke_report_2023-06-01_2025-12-31.csv") # Scraped
MASS_FILE = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-18_du-0.5_qa-0_sza-70_2023-06-01_2023-12-31.csv")  # Generated from Mass Integration of TROPOMI SO2 L2 Product (Phase 1 in sensitivty_analysis.py)

# --------------------------------------------------------------------------- Define Date Range -----------------------------------------------------------------------------------
START_DATE = '2023-06-01'
END_DATE = '2023-12-31'

# ==============================================================================
# 2. DATA LOADING & PROCESSING
# ==============================================================================

# --- Plot 1: Flux Data ---
df_ton = pd.read_csv(GROUND_TON_FILE)
df_ton['date'] = pd.to_datetime(df_ton['date'])

df_flux = pd.read_csv(SAT_FLUX_FILE, comment='#')
df_flux['time'] = pd.to_datetime(df_flux['time']).dt.tz_localize(None)

# Filter bounds
df_ton_filt = df_ton[(df_ton['date'] >= START_DATE) & (df_ton['date'] <= END_DATE)].sort_values('date')
df_flux_filt = df_flux[(df_flux['time'] >= START_DATE) & (df_flux['time'] <= END_DATE)].sort_values('time')

# Add Date Only for Merging
df_ton_filt['date_only'] = df_ton_filt['date'].dt.date
df_flux_filt['date_only'] = df_flux_filt['time'].dt.date

# Adding Missing Data Proportion 
# Load file, skipping the metadata header line
df_mass = pd.read_csv(MASS_FILE, skiprows=1)
# Use dayfirst=True to handle dd/mm/yyyy format properly
df_mass['Time'] = pd.to_datetime(df_mass['Time'])
df_mass['date_only'] = df_mass['Time'].dt.date

# Filter for the specific radii needed to evaluate missing logic
df_mass_25 = df_mass[df_mass['radius'] == 25]
df_mass_500 = df_mass[df_mass['radius'] == 500]

# Identify bad dates: radius 25km > 50% OR radius 500km > 20%
bad_25_dates = df_mass_25[df_mass_25['proportion_missing'] > 50]['date_only'].unique()
bad_500_dates = df_mass_500[df_mass_500['proportion_missing'] > 20]['date_only'].unique()
unreliable_dates = set(bad_25_dates).union(set(bad_500_dates))

# Flag unreliable points in the flux dataframe
df_flux_filt['is_unreliable'] = df_flux_filt['date_only'].apply(lambda d: d in unreliable_dates)

# Split the flux data for plotting
df_flux_reliable = df_flux_filt[~df_flux_filt['is_unreliable']]
df_flux_unreliable = df_flux_filt[df_flux_filt['is_unreliable']]

# Processing cloud file
df_cloud = pd.read_csv(CLOUD_FILE, comment='#')
df_cloud['Time'] = pd.to_datetime(df_cloud['Time'])

# Filter for 50km radius and a cloud fraction threshold
CLOUD_THRESHOLD = 0.25
cloudy_days = df_cloud[(df_cloud['radius'] == 500) & 
                       (df_cloud['TROPOMI_CloudFraction_mean'] > CLOUD_THRESHOLD)]['Time'].dt.date.unique()

# --- Plot 2: Thermal Radiance ---
df_viirs = pd.read_csv(THERMAL_FILE)
df_viirs['date'] = pd.to_datetime(df_viirs['acq_date'])
# Filter for type 1
df_viirs_type1 = df_viirs[df_viirs['type'] == 1]
# Filter by date range
df_viirs_filt = df_viirs_type1[(df_viirs_type1['date'] >= START_DATE) & (df_viirs_type1['date'] <= END_DATE)]

# --- Plot 3: Seismic Activity ---
df_seismic = pd.read_csv(SEISMIC_FILE)
df_seismic['date'] = pd.to_datetime(df_seismic['event_date'])
df_seismic_filt = df_seismic[(df_seismic['date'] >= START_DATE) & (df_seismic['date'] <= END_DATE)].sort_values('date')

# Map new acronym columns to descriptive names for the plot legend
seismic_col_map = {
    'VTA': 'Deep Volcanic (VTA)',
    'VTB': 'Shallow Volcanic (VTB)',
    'MP': 'Multi-Phase (MP)',
    'LF': 'Low Frequency (LF)',
    'TREM': 'Tremor (TREM)',
    'TECT': 'Tectonic (TECT)',
    'RF': 'Rockfall (RF)',
    'AP': 'Pyroclastic Flow (AP)'
}
# Extract just the keys for plot
seismic_cols = list(seismic_col_map.keys())

# --- Plot 4: Volcanic Activity ---
df_volc = pd.read_csv(VOLCANIC_FILE)
# Parse weekly dates to end date for plotting
df_volc['end_date_str'] = df_volc['date'].apply(lambda x: x.split('-')[1].strip() if '-' in str(x) else x)
df_volc['parsed_date'] = pd.to_datetime(df_volc['end_date_str'])
df_volc_filt = df_volc[(df_volc['parsed_date'] >= START_DATE) & (df_volc['parsed_date'] <= END_DATE)].sort_values('parsed_date')

# --- Plot 5: Smoke Report ---
df_smoke = pd.read_csv(SMOKE_FILE)
df_smoke['date'] = pd.to_datetime(df_smoke['date'])
df_smoke_filt = df_smoke[(df_smoke['date'] >= START_DATE) & (df_smoke['date'] <= END_DATE)].sort_values('date')

# Map descriptive smoke intensity to numerical values for plotting
intensity_map = {'tipis': 1, 'sedang': 2, 'tebal': 3}
df_smoke_filt['intensity_num'] = df_smoke_filt['smoke_intensity'].map(intensity_map).fillna(0)

# ==============================================================================
# 3. FIGURE CREATION (5 Subplots)
# ==============================================================================
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(18, 18), sharex=True, constrained_layout=True)
fig.suptitle("Mt. Merapi Multi-Parameter Monitoring Data: June - Dec 2023", fontsize=24)

plt.rcParams.update({
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})
# ------------------------------------------------------------------------------
# PLOT 1: Ground + Satellite Flux
# ------------------------------------------------------------------------------
color1 = 'tab:blue'
ax1.plot(df_ton_filt['date'], df_ton_filt['observed_x2.4_ton'], color=color1, marker='o', label='Ground Tonnage')

# 1. Create a Proxy Patch for the legend (this doesn't get plotted on the axis)
cloud_patch = mpatches.Patch(color='gray', alpha=0.15, label=f'High Cloud (> {CLOUD_THRESHOLD*100:.0f}%)')

# 2. Draw the spans WITHOUT labels to avoid legend clutter
# Ensure unique dates to prevent redundant over-plotting
unique_cloudy_days = sorted(list(set(cloudy_days)))

for c_date in unique_cloudy_days:
    ax1.axvspan(pd.Timestamp(c_date) - pd.Timedelta(hours=12), 
                pd.Timestamp(c_date) + pd.Timedelta(hours=12), 
                color='gray', alpha=0.15, zorder=0)

ax1.set_ylabel('Observed Tonnage x2.4\n(tons)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

ax1_twin = ax1.twinx()
ax1_twin.plot(df_flux_filt['time'], df_flux_filt['flux'] * 1000, color='darkorange', linestyle='--', alpha=0.6)
ax1_twin.scatter(df_flux_reliable['time'], df_flux_reliable['flux'] * 1000, color='darkorange', marker='o', label='Sat Flux (Reliable)')
ax1_twin.scatter(df_flux_unreliable['time'], df_flux_unreliable['flux'] * 1000, color='red', marker='x', label='Sat Flux (Unreliable)')

ax1_twin.set_ylabel('Satellite Flux\n(tons)', color='darkorange')
ax1_twin.tick_params(axis='y', labelcolor='darkorange')

# 3. Simple Legend Construction
# Collect handles from both axes and manually append our proxy patch
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()

# 2. Add the legend to ax1_twin (the layer that sits on top)
leg = ax1_twin.legend(lines1 + lines2 + [cloud_patch], 
                      labels1 + labels2 + [cloud_patch.get_label()], 
                      loc='upper left', frameon=True)

# 3. Force the styling
leg.set_zorder(100)
frame = leg.get_frame()
frame.set_facecolor('white')
frame.set_alpha(1.0) # Set to opaque
frame.set_edgecolor('gray')

ax1.set_title("1. SO$_2$ Flux (Ground-Based vs Satellite-Derived)",fontsize=18)
# ------------------------------------------------------------------------------
# PLOT 2: Thermal Radiance
# ------------------------------------------------------------------------------
# Draw vertical lines from 0 to the FRP values
ax2.vlines(x=df_viirs_filt['date'], ymin=0, ymax=df_viirs_filt['frp'], color='orange', alpha=0.5, linewidth=1.5)

# Plot the scatter points on top of the lines (zorder=3 puts them above the grid/lines)
ax2.scatter(df_viirs_filt['date'], df_viirs_filt['frp'], color='darkorange', edgecolor='k', zorder=3, label='Fire Radiative Power (MW)')

ax2.set_ylabel("Thermal Radiative Power (MW)")
ax2.set_ylim(bottom=0)  # Anchor the y-axis exactly at zero
ax2.legend(loc='upper left', frameon=True)
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.set_title("2. Thermal Activity", fontsize=18)

# ------------------------------------------------------------------------------
# PLOT 3: Seismic Activity
# ------------------------------------------------------------------------------
bottom_vals = np.zeros(len(df_seismic_filt))
for col in seismic_cols:
    if col in df_seismic_filt.columns:
        # Fill missing values with 0 so the stacked bar chart doesn't break
        vals = df_seismic_filt[col].fillna(0).values
        # Use the mapped, readable name for the legend label
        ax3.bar(df_seismic_filt['date'], vals, bottom=bottom_vals, label=seismic_col_map[col], 
                width=1.0, edgecolor='white', linewidth=0.5)
        bottom_vals += vals

ax3.set_ylabel("Daily Event Count")
# Set ncol=3 to accommodate the 9 new legend items nicely
ax3.legend(loc='upper left', ncol=2)
ax3.grid(True, linestyle=':', alpha=0.5)
ax3.set_title("3. Seismic Activity", fontsize=20)


# ------------------------------------------------------------------------------
# PLOT 4: Volcanic Activity 
# ------------------------------------------------------------------------------
# Plot Avalanche Count (Bars)
ax4.bar(df_volc_filt['parsed_date'], df_volc_filt['avalanche_count'], width=4.0, color='gray', alpha=0.6, label='Avalanche Count')
ax4.set_ylabel('Avalanche Count', color='gray')
ax4.tick_params(axis='y', labelcolor='gray')

# Plot Pyroclastic flow on a secondary axis (Lines/Markers)
ax4_twin = ax4.twinx()
ax4_twin.plot(df_volc_filt['parsed_date'], df_volc_filt['pyroclastic_flow_count'], color='purple', marker='D', markersize=6, linestyle='-', label='Pyroclastic Flow Count')
ax4_twin.set_ylabel('Pyroclastic Flow Count', color='purple')
ax4_twin.tick_params(axis='y', labelcolor='purple')

# Indicate Ashfall as red vertical spans over the background
ashfall_dates = df_volc_filt[df_volc_filt['ashfall_occurrence'] == True]['parsed_date']
for date in ashfall_dates:
    ax4.axvspan(date - pd.Timedelta(days=3), date + pd.Timedelta(days=3), color='red', alpha=0.15)

# Create a manual Patch for the legend instead of plotting NaT
ashfall_patch = mpatches.Patch(color='red', alpha=0.15, label='Ashfall Occurrence Window')

# Grab the existing lines and labels from both the primary and twin axes
lines4, labels4 = ax4.get_legend_handles_labels()
lines4_twin, labels4_twin = ax4_twin.get_legend_handles_labels()

# Combine them all into the final legend
leg = ax4_twin.legend(lines4 + lines4_twin + [ashfall_patch],
                 labels4 + labels4_twin + ['Ashfall Occurrence Window'],
                 loc='upper left',
                 frameon=True)

# Force the styling
leg.set_zorder(1000)
frame = leg.get_frame()
frame.set_facecolor('white')
frame.set_alpha(1.0) # Set to opaque
frame.set_edgecolor('gray')

ax4.grid(True, linestyle=':', alpha=0.5)
ax4.set_title("4. Volcanic Activity (Weekly Avalanches, Pyroclastic Flow & Ashfall)", fontsize=18)

# ------------------------------------------------------------------------------
# PLOT 5: Smoke Visibility 
# ------------------------------------------------------------------------------
ax5.step(df_smoke_filt['date'], df_smoke_filt['intensity_num'], where='mid', color='slategray', linewidth=2)
ax5.fill_between(df_smoke_filt['date'], df_smoke_filt['intensity_num'], step='mid', color='slategray', alpha=0.3)

ax5.set_yticks([0, 1, 2, 3])
ax5.set_yticklabels(['None', 'Thin', 'Medium', 'Thick'])
ax5.set_ylabel("Smoke Intensity")
ax5.grid(True, linestyle=':', alpha=0.5)
ax5.set_title("5. Daily Smoke Visibility / Intensity", fontsize=18)

# ==============================================================================
# 4. FINAL FORMATTING & DISPLAY
# ==============================================================================
ax5.set_xlim(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m-%y'))
ax5.xaxis.set_major_locator(mdates.MonthLocator())
ax5.set_xlabel("Date (mm-yy)")

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
## Figure 5: Combined Plots of SO2 + Thermal + Seismic + Volcanic Activity (2024-25)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches  
from scipy.stats import norm

# ==============================================================================
# 1. FILE PATHS & CONFIGURATION
# ==============================================================================
# ------------------------------------------------------------------------Add Local Files Paths here ------------------------------------------------------------------------------
# Note that the output_0.8.csv (GROUND_TON_FILE) and bpptkg_seismicity_counts.csv (SEISMIC_FILE) files are not provided in the "Figure 4" folder since those are data provided by BPPTKG.
# These data are available 1from BPPTKG upon request. However, sample files are included to indicate how the format of these files would look like.
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Figure 5")  # Navigate to folder with the data files

GROUND_TON_FILE = os.path.join(data_dir, "output_0.8.csv")
SAT_FLUX_FILE = os.path.join(data_dir, "Merapi_Flux_7km_2024-01-02_2025-12-31_tracks-0_du-0.3_qa-0_sza-70_drops-none_P550.csv")
CLOUD_FILE = os.path.join(data_dir, "TROPOMI_CloudFraction_mean_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-0_qa-0_sza-70_2024-01-02_2025-12-31.csv")
THERMAL_FILE = os.path.join(data_dir, "fire_archive_J1V-C2_733161.csv")
SEISMIC_FILE = os.path.join(data_dir, "bpptkg_seismicity_counts.csv")
VOLCANIC_FILE = os.path.join(data_dir, "merapi_weekly_metrics_24_25.csv")
SMOKE_FILE = os.path.join(data_dir, "merapi_smoke_report_2023-06-01_2025-12-31.csv")
MASS_FILE = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-0_du-0.3_qa-0_sza-70_2024-01-02_2025-12-31.csv")

# --------------------------------------------------------------------------- Define Date Range -----------------------------------------------------------------------------------
START_DATE = '2024-01-01'
END_DATE = '2025-12-31'

# ==============================================================================
# 2. DATA LOADING & PROCESSING
# ==============================================================================

# --- Plot 1: Flux Data ---
df_ton = pd.read_csv(GROUND_TON_FILE)
df_ton['date'] = pd.to_datetime(df_ton['date'])

df_flux = pd.read_csv(SAT_FLUX_FILE, comment='#')
df_flux['time'] = pd.to_datetime(df_flux['time']).dt.tz_localize(None)

# Filter bounds
df_ton_filt = df_ton[(df_ton['date'] >= START_DATE) & (df_ton['date'] <= END_DATE)].sort_values('date')
df_flux_filt = df_flux[(df_flux['time'] >= START_DATE) & (df_flux['time'] <= END_DATE)].sort_values('time')

# Add Date Only for Merging
df_ton_filt['date_only'] = df_ton_filt['date'].dt.date
df_flux_filt['date_only'] = df_flux_filt['time'].dt.date

# Adding Missing Data Proportion 
# Load file, skipping the metadata header line
df_mass = pd.read_csv(MASS_FILE, skiprows=1)
# Use dayfirst=True to handle dd/mm/yyyy format properly
df_mass['Time'] = pd.to_datetime(df_mass['Time'])
df_mass['date_only'] = df_mass['Time'].dt.date

# Filter for the specific radii needed to evaluate missing logic
df_mass_25 = df_mass[df_mass['radius'] == 25]
df_mass_500 = df_mass[df_mass['radius'] == 500]

# Identify bad dates: radius 25km > 50% OR radius 500km > 20%
bad_25_dates = df_mass_25[df_mass_25['proportion_missing'] > 50]['date_only'].unique()
bad_500_dates = df_mass_500[df_mass_500['proportion_missing'] > 20]['date_only'].unique()
unreliable_dates = set(bad_25_dates).union(set(bad_500_dates))

# Flag unreliable points in the flux dataframe
df_flux_filt['is_unreliable'] = df_flux_filt['date_only'].apply(lambda d: d in unreliable_dates)

# Split the flux data for plotting
df_flux_reliable = df_flux_filt[~df_flux_filt['is_unreliable']]
df_flux_unreliable = df_flux_filt[df_flux_filt['is_unreliable']]

# Processing cloud file
df_cloud = pd.read_csv(CLOUD_FILE, comment='#')
df_cloud['Time'] = pd.to_datetime(df_cloud['Time'])

# Filter for 50km radius and a cloud fraction threshold
CLOUD_THRESHOLD = 0.25
cloudy_days = df_cloud[(df_cloud['radius'] == 500) & 
                       (df_cloud['TROPOMI_CloudFraction_mean'] > CLOUD_THRESHOLD)]['Time'].dt.date.unique()

# --- Plot 2: Thermal Radiance ---
df_viirs = pd.read_csv(THERMAL_FILE)
df_viirs['date'] = pd.to_datetime(df_viirs['acq_date'])
# Filter for type 1
df_viirs_type1 = df_viirs[df_viirs['type'] == 1]
# Filter by date range
df_viirs_filt = df_viirs_type1[(df_viirs_type1['date'] >= START_DATE) & (df_viirs_type1['date'] <= END_DATE)]

# --- Plot 3: Seismic Activity ---
df_seismic = pd.read_csv(SEISMIC_FILE)
df_seismic['date'] = pd.to_datetime(df_seismic['event_date'])
df_seismic_filt = df_seismic[(df_seismic['date'] >= START_DATE) & (df_seismic['date'] <= END_DATE)].sort_values('date')

# Map new acronym columns to descriptive names for the plot legend
seismic_col_map = {
    'VTA': 'Deep Volcanic (VTA)',
    'VTB': 'Shallow Volcanic (VTB)',
    'MP': 'Multi-Phase (MP)',
    'LF': 'Low Frequency (LF)',
    'TREM': 'Tremor (TREM)',
    'TECT': 'Tectonic (TECT)',
    'RF': 'Rockfall (RF)',
    'AP': 'Pyroclastic Flow (AP)'
}
# Extract just the keys for plot
seismic_cols = list(seismic_col_map.keys())

# --- Plot 4: Volcanic Activity ---
df_volc = pd.read_csv(VOLCANIC_FILE)
# Parse weekly dates to end date for plotting
df_volc['end_date_str'] = df_volc['date'].apply(lambda x: x.split('-')[1].strip() if '-' in str(x) else x)
df_volc['parsed_date'] = pd.to_datetime(df_volc['end_date_str'])
df_volc_filt = df_volc[(df_volc['parsed_date'] >= START_DATE) & (df_volc['parsed_date'] <= END_DATE)].sort_values('parsed_date')

# ==============================================================================
# 3. FIGURE CREATION (4 Subplots)
# ==============================================================================
fig, (ax1, ax2, ax3, ax4) = plt.subplots(
    4, 1, figsize=(22, 18), sharex=True, constrained_layout=True
)
# Update title to match the 2-year span
fig.suptitle("Mt. Merapi Multi-Parameter Monitoring Data: Jan 2024 - Dec 2025", fontsize=26)

plt.rcParams.update({
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16
})

# ------------------------------------------------------------------------------
# PLOT 1: Ground + Satellite Flux
# ------------------------------------------------------------------------------
color1 = 'tab:blue'
ax1.plot(df_ton_filt['date'], df_ton_filt['observed_x2.4_ton'], color=color1, marker='o', label='Ground Tonnage')

# 1. Create a Proxy Patch for the legend (this doesn't get plotted on the axis)
cloud_patch = mpatches.Patch(color='gray', alpha=0.15, label=f'High Cloud (> {CLOUD_THRESHOLD*100:.0f}%)')

# 2. Draw the spans WITHOUT labels to avoid legend clutter
# Ensure unique dates to prevent redundant over-plotting
unique_cloudy_days = sorted(list(set(cloudy_days)))

for c_date in unique_cloudy_days:
    ax1.axvspan(pd.Timestamp(c_date) - pd.Timedelta(hours=12), 
                pd.Timestamp(c_date) + pd.Timedelta(hours=12), 
                color='gray', alpha=0.15, zorder=0)

ax1.set_ylabel('Observed Tonnage x2.4\n(tons)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.grid(True, alpha=0.3)

ax1_twin = ax1.twinx()
ax1_twin.plot(df_flux_filt['time'], df_flux_filt['flux'] * 1000, color='darkorange', linestyle='--', alpha=0.6)
ax1_twin.scatter(df_flux_reliable['time'], df_flux_reliable['flux'] * 1000, color='darkorange', marker='o', label='Sat Flux (Reliable)')
ax1_twin.scatter(df_flux_unreliable['time'], df_flux_unreliable['flux'] * 1000, color='red', marker='x', label='Sat Flux (Unreliable)')

ax1_twin.set_ylabel('Satellite Flux\n(tons)', color='darkorange')
ax1_twin.tick_params(axis='y', labelcolor='darkorange')

# 3. Simple Legend Construction
# Collect handles from both axes and manually append our proxy patch
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()

# 2. Add the legend to ax1_twin (the layer that sits on top)
leg = ax1_twin.legend(lines1 + lines2 + [cloud_patch], 
                      labels1 + labels2 + [cloud_patch.get_label()], 
                      loc='upper left', 
                      frameon=True)

# 3. Force the styling
leg.set_zorder(100)
frame = leg.get_frame()
frame.set_facecolor('white')
frame.set_alpha(1.0) # Set to opaque
frame.set_edgecolor('gray')

ax1.set_title("1. SO$_2$ Flux (Ground-Based vs Satellite-Derived)",fontsize=20)
# ------------------------------------------------------------------------------
# PLOT 2: Thermal Radiance
# ------------------------------------------------------------------------------
# Draw vertical lines from 0 to the FRP values
ax2.vlines(x=df_viirs_filt['date'], ymin=0, ymax=df_viirs_filt['frp'], color='orange', alpha=0.5, linewidth=1.5)

# Plot the scatter points on top of the lines (zorder=3 puts them above the grid/lines)
ax2.scatter(df_viirs_filt['date'], df_viirs_filt['frp'], color='darkorange', edgecolor='k', zorder=3, label='Fire Radiative Power (MW)')

ax2.set_ylabel("Thermal Radiative Power (MW)")
ax2.set_ylim(bottom=0)  # Anchor the y-axis exactly at zero
ax2.legend(loc='upper left', frameon=True)
ax2.grid(True, linestyle=':', alpha=0.5)
ax2.set_title("2. Thermal Activity", fontsize=20)

# ------------------------------------------------------------------------------
# PLOT 3: Seismic Activity
# ------------------------------------------------------------------------------
bottom_vals = np.zeros(len(df_seismic_filt))
for col in seismic_cols:
    if col in df_seismic_filt.columns:
        # Fill missing values with 0 so the stacked bar chart doesn't break
        vals = df_seismic_filt[col].fillna(0).values
        # Use the mapped, readable name for the legend label
        ax3.bar(df_seismic_filt['date'], vals, bottom=bottom_vals, label=seismic_col_map[col], 
                width=1.0, edgecolor='white', linewidth=0.5)
        bottom_vals += vals

ax3.set_ylabel("Daily Event Count")
# Set ncol=3 to accommodate the 9 new legend items nicely
ax3.legend(loc='upper left', ncol=2)
ax3.grid(True, linestyle=':', alpha=0.5)
ax3.set_title("3. Seismic Activity", fontsize=20)

# ------------------------------------------------------------------------------
# PLOT 4: Volcanic Activity 
# ------------------------------------------------------------------------------
# Plot Avalanche Count (Bars)
ax4.bar(df_volc_filt['parsed_date'], df_volc_filt['avalanche_count'], width=4.0, color='gray', alpha=0.6, label='Avalanche Count')
ax4.set_ylabel('Avalanche Count', color='gray')
ax4.tick_params(axis='y', labelcolor='gray')

# Plot Pyroclastic flow on a secondary axis (Lines/Markers)
ax4_twin = ax4.twinx()
ax4_twin.plot(df_volc_filt['parsed_date'], df_volc_filt['pyroclastic_flow_count'], color='purple', marker='D', markersize=6, linestyle='-', label='Pyroclastic Flow Count')
ax4_twin.set_ylabel('Pyroclastic Flow Count', color='purple')
ax4_twin.tick_params(axis='y', labelcolor='purple')

# Indicate Ashfall as red vertical spans over the background
ashfall_dates = df_volc_filt[df_volc_filt['ashfall_occurrence'] == True]['parsed_date']
for date in ashfall_dates:
    ax4.axvspan(date - pd.Timedelta(days=3), date + pd.Timedelta(days=3), color='red', alpha=0.15)

# Create a manual Patch for the legend instead of plotting NaT
ashfall_patch = mpatches.Patch(color='red', alpha=0.15, label='Ashfall Occurrence Window')

# Grab the existing lines and labels from both the primary and twin axes
lines4, labels4 = ax4.get_legend_handles_labels()
lines4_twin, labels4_twin = ax4_twin.get_legend_handles_labels()

# Combine them all into the final legend
leg = ax4_twin.legend(lines4 + lines4_twin + [ashfall_patch],
                 labels4 + labels4_twin + ['Ashfall Occurrence Window'],
                 loc='upper left',
                 frameon=True)

# Force the styling
leg.set_zorder(1000)
frame = leg.get_frame()
frame.set_facecolor('white')
frame.set_alpha(1.0) # Set to opaque
frame.set_edgecolor('gray')

ax4.grid(True, linestyle=':', alpha=0.5)
ax4.set_title("4. Volcanic Activity (Weekly Avalanches, Pyroclastic Flow & Ashfall)", fontsize=20)
# ==============================================================================
# 4. FINAL FORMATTING & DISPLAY
# ==============================================================================
year_boundary = pd.to_datetime('2025-01-01')
for ax in [ax1, ax2, ax3, ax4]:
    ax.axvline(year_boundary, color='black', linestyle='--', alpha=0.4, linewidth=1.5, zorder=1)

ax4.set_xlim(pd.to_datetime(START_DATE), pd.to_datetime(END_DATE))
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m-%y'))
ax4.xaxis.set_major_locator(mdates.MonthLocator()) 
ax4.set_xlabel("Date (mm-yy)")

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX_25.png", dpi=400)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
## Simulation of Synthetic Plumes for Sensitivity Analysis of SO2 Flux Retrievals (Both Degassing)
import os, json, numpy as np, pandas as pd
import subprocess
import itertools

# ------------------------- Things to Change: -------------------------
# Simulation Grids 
# Atmospheric Parameters and Flux Rates in the generate_synthetic_phase1 function
# Wind speed in the run_phase2_inversion function (should match the one used in generation)
# true_flux in the execution loop (should match the flux_target used in generation)
# ---------------------------------------------------------------------

# 1. CONFIGURATION & PATHS
# ------------------------------------------------------------------------ INSERT PATH FOR FOLDERS HERE -------------------------------------------------------------------------
OUTPUT_DIR = r"XXX\synthetic\both"  # Folder for storing the CSV outputs used for the Flux Calculator tool
TS_DIR = r"XXX\synthetic\both"  # Folder for storing the time series outputs from the Flux Calculator tool
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TS_DIR, exist_ok=True)

RADII = [500, 400, 300, 250, 200, 150, 100, 75, 50, 25]
KAPPA = 0.0007 # Grandin's conversion factor for 25km^2 pixels

# Simulation Grids to test
# Note that wind direction is in the +ve x direction, so 0 degrees means the interfering volcano is directly downwind,
# 90 degrees means it's crosswind, and 180 degrees means it's directly upwind relative to the target volcano.
DISTANCES_GRID = [25, 50, 100, 150, 200] # km away from target
ANGLES_GRID = [0, 45, 90, 135, 180] # Degrees relative to wind (0 = directly downwind, 90 = crosswind, 180 = directly upwind)
DU_THRESHOLDS = [0.3] # Truncation threshold; assume perfect knowledge to isolate plume effects
INTERFERING_FLUXES = [0, 0.5, 1.0, 1.5] # Varying neighbouring volcano fluxes (kton/day)

# 2. SYNTHETIC PLUME GENERATION

def calculate_plume_du(X, Y, src_x, src_y, flux_kton_day, u, Dy, k):
    """
    Evaluates Eq 6. directly outputting column amounts in DU.
    Note: A scaling factor is often needed to translate abstract kton/day 
    into realistic DU values based on the grid resolution.
    """
    # --- UNIT CONVERSIONS ---
    # Eq 6 needs mass flux in kg/s to match u (m/s) and Dy (m^2/s)
    # 1 kton = 1,000,000 kg. 1 day = 86,400 seconds.
    flux_kg_s = flux_kton_day * (1e6 / 86400.0)

    x_local = X - src_x
    y_local = Y - src_y
    
    D = np.zeros_like(X)
    mask = x_local > 0 # Plume only flows downwind
    
    x_valid = x_local[mask]
    y_valid = y_local[mask]
    
    # Physics components (Eq 6)
    coeff = (flux_kg_s / u) / np.sqrt(4 * np.pi * Dy * (x_valid / u))
    dispersion = np.exp((-u * y_valid**2) / (4 * Dy * x_valid))
    decay = np.exp((-k * x_valid) / u)
    
    # Calculate mass field in kg/m^2
    mass_field_kg_m2 = coeff * dispersion * decay
    
    # CONVERT TO DOBSON UNITS (DU)
    # 1 DU is approximately 2.857e-5 kg/m^2 for SO2
    DU_CONVERSION = 2.857e-5 
    D[mask] = mass_field_kg_m2 / DU_CONVERSION 
    
    return D

def generate_synthetic_phase1(distance_km, angle_deg, du_threshold, flux_interfere):
    """
    Replaces your run_phase1_extraction. Generates the plumes, integrates them,
    and saves the CSVs required by the Flux Calculator tool.
    """
    # 1. Setup 2D Grid (in meters, roughly matching 0.05 deg resolution)
    # Using 5000m spacing to match A = 25km^2
    x_coords = np.arange(-200e3, 800e3, 5000) 
    y_coords = np.arange(-500e3, 500e3, 5000)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    # Calculate distance from origin (Target Volcano) for every pixel in km
    grid_dist_km = np.sqrt(X**2 + Y**2) / 1000.0

    # 2. Atmospheric & Source Parameters
    u = 5.0        # Wind speed m/s (Blows along positive X axis)
    Dy = 1e4         # Cross-plume diffusivity m^2/s
    k = 1e-5         # Gas loss rate 1/s
    sigma_CA = 0.3   # Background noise in DU
    
    flux_target = 1.0   # Target volcano flux (kton/day)

    # 3. Position Interfering Volcano using polar coordinates
    distance_m = distance_km * 1000.0
    theta_rad = np.radians(angle_deg)
    src2_x = distance_m * np.cos(theta_rad)
    src2_y = distance_m * np.sin(theta_rad)

    # 4. Generate Plumes + Noise
    target_plume = calculate_plume_du(X, Y, 0, 0, flux_target, u, Dy, k)
    neighbour_plume = calculate_plume_du(X, Y, src2_x, src2_y, flux_interfere, u, Dy, k)
    
    noise = np.random.normal(0, sigma_CA, X.shape)
    total_du = target_plume + neighbour_plume + noise

    # 5. Apply Truncation (Your 'grid_so2_final' logic)
    total_du[total_du < du_threshold] = 0.0

    # 6. Spatial Integration (Mimicking your for-loop over CONFIG['radii'])
    so2_results = []
    cf_results = []
    
    # Simulate 3 consecutive days to satisfy any time-series array requirements in the Flux Calculator tool
    sim_times = [
        "2023-01-01 12:00:00",
        "2023-01-02 12:00:00",
        "2023-01-03 12:00:00"
    ]

    for sim_time in sim_times:
        for r in RADII:
            # Mask pixels within radius
            in_circle = grid_dist_km <= r
            pixels_in_circle = total_du[in_circle]
            
            so2_sum = (pixels_in_circle * KAPPA).sum()
            npoints = (pixels_in_circle > 0).sum()
            
            # Compile row identical to real script
            metrics = {
                "Time": sim_time,
                "TROPOMI_SO2-7km_mass": so2_sum,
                "radius": r,
                "npoints": npoints,
                "proportion_missing": 0.0, # Perfect synthetic grid
                "centroid_lon": 110.446,   # Dummy coordinates
                "centroid_lat": -7.54,
                "fraction_filtered_edge_tracks": 0.0,
                "fraction_filtered_quality": 0.0,
                "fraction_filtered_sza": 0.0,
                "fraction_filtered_total": 0.0
            }
            so2_results.append(metrics)
            
            # Dummy cloud data as the Flux Calculator tool requires this file to exist for processing
            cf_metrics = metrics.copy()
            del cf_metrics["TROPOMI_SO2-7km_mass"]
            cf_metrics["TROPOMI_CloudFraction_mean"] = 0.0 
            cf_results.append(cf_metrics)

    # 7. Save to CSVs WITH the Metadata Header
    rad_str = "-".join(map(str, sorted(RADII))) + "km"
    label = f"Dist{distance_km}_Ang{angle_deg}_DU{du_threshold}_IntFlux{flux_interfere}"
    
    fn_so2 = os.path.join(OUTPUT_DIR, f"TROPOMI_SO2-7km_mass_{rad_str}_Synthetic_{label}.csv")
    fn_cf = os.path.join(OUTPUT_DIR, f"TROPOMI_CloudFraction_mean_{rad_str}_Synthetic_{label}.csv")
    
    # Replicate the metadata header from your Phase 1 script
    dummy_meta = {
        "volcano": "Synthetic_Merapi",
        "latitude": -7.54, "longitude": 110.446,
        "radii": RADII,
        "filters": {"tracks": 0, "du": du_threshold, "qa": 0, "sza": 70},
        "start": "2023-01-01", "end": "2023-01-03",
        "grid_res": 0.05, "max_gap_deg": 1.0 
    }
    
    with open(fn_so2, 'w', newline='') as f:
        so2_meta = {"variable": "TROPOMI_SO2-7km_mass", **dummy_meta}
        f.write(f"# Computation info: {json.dumps(so2_meta)}\n")
        pd.DataFrame(so2_results).to_csv(f, index=False, lineterminator='\n')

    with open(fn_cf, 'w', newline='') as f:
        cf_meta = {"variable": "TROPOMI_CloudFraction_mean", **dummy_meta}
        f.write(f"# Computation info: {json.dumps(cf_meta)}\n")
        pd.DataFrame(cf_results).to_csv(f, index=False, lineterminator='\n')
        
    return fn_so2, fn_cf

# 3. PHASE 2 INVERSION (Flux Calculator TOOL)

def run_phase2_inversion(so2_csv, cf_csv, label):
    """
    Directly runs the Flux Calculator tool on the synthetic CSVs.
    """
    output_filename = f"Synthetic_Flux_{label}"
    
    # Use the same constant wind speed here to match the simulation 'u = 5.0'
    command = [
        "so2_flux_calculator",
        so2_csv.replace("\\", "/"),
        cf_csv.replace("\\", "/"),
        "--method", "curvefit",
        "--wind_speed", "5",
        "--output_file", output_filename,
        "--output_directory", TS_DIR.replace("\\", "/")
    ]
    
    try:
        # We pass check=False so it doesn't halt the whole script, but we capture the error
        result = subprocess.run(command, capture_output=True, text=True, shell=True)
        
        # If the Flux Calculator tool crashed, print exactly why it crashed
        if result.returncode != 0:
            print(f"\n[!] The Flux Calculator tool crashed on {label} [!]")
            print(result.stderr)
            print("-" * 40 + "\n")
            
    except Exception as e:
        print(f"Subprocess failed to launch: {e}")
        
    return os.path.join(TS_DIR, output_filename + ".csv")

# 4. EXECUTION LOOP

if __name__ == '__main__':
    all_combinations = list(itertools.product(DISTANCES_GRID, ANGLES_GRID, DU_THRESHOLDS, INTERFERING_FLUXES))
    print(f"Starting Simulation of {len(all_combinations)} scenarios...")
    
    results_log = []
    true_flux = 1.0 # The flux_target we set in the generation function
    
    for dist, ang, du, int_flux in all_combinations:
        label = f"Dist{dist}_Ang{ang}_DU{du}_IntFlux{int_flux}"
        print(f"Processing: {label}")
        
        # 1. Generate Synthetic Grid and Integrate
        so2_path, cf_path = generate_synthetic_phase1(dist, ang, du, int_flux)
        
        # 2. Run CLI Inversion
        ts_path = run_phase2_inversion(so2_path, cf_path, label)
        
        # 3. Evaluate Performance
        try:
            tropomi_df = pd.read_csv(ts_path, comment='#')
            # Extract the retrieved flux and uncertainty (1 row since only 1 day was simulated)
            retrieved_flux = tropomi_df['flux'].iloc[0]
            uncertainty = tropomi_df['sigma_flux'].iloc[0]
            
            error = retrieved_flux - true_flux
            
            results_log.append({
                'Distance_km': dist,
                'Angle_deg': ang,
                'DU_Threshold': du,
                'Interfering_Flux': int_flux,
                'True_Flux': true_flux,
                'Retrieved_Flux': retrieved_flux,
                'Error': error,
                'Reported_Uncertainty': uncertainty
            })
        except Exception as e:
            print(f"Could not read output for {label}: {e}")

    # Save final sensitivity analysis
    df_results = pd.DataFrame(results_log)
    df_results.to_csv(os.path.join(TS_DIR, "Synthetic_Sensitivity_Results_Both_Degas.csv"), index=False)
    print("\nSimulation Complete. Results saved.")

In [ ]:
## Simulation of Synthetic Plumes for Sensitivity Analysis of SO2 Flux Retrievals (Target Not Degassing)
import os, json, numpy as np, pandas as pd
import subprocess
import itertools

# ------------------------- Things to Change: -------------------------
# Simulation Grids 
# Atmospheric Parameters and Flux Rates in the generate_synthetic_phase1 function
# Wind speed in the run_phase2_inversion function (should match the one used in generation)
# true_flux in the execution loop (should match the flux_target used in generation)
# ---------------------------------------------------------------------

# 1. CONFIGURATION & PATHS
# ------------------------------------------------------------------------ INSERT PATH FOR FOLDERS HERE -------------------------------------------------------------------------
OUTPUT_DIR = r"XXX\synthetic\neighbourdegas"  # Folder for storing the CSV outputs used for the Flux Calculator tool
TS_DIR = r"XXX\synthetic\neighbourdegas"  # Folder for storing the time series outputs from the Flux Calculator tool
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TS_DIR, exist_ok=True)

RADII = [500, 400, 300, 250, 200, 150, 100, 75, 50, 25]
KAPPA = 0.0007 # Grandin's conversion factor for 25km^2 pixels

# Simulation Grids to test
# Note that wind direction is in the +ve x direction, so 0 degrees means the interfering volcano is directly downwind,
# 90 degrees means it's crosswind, and 180 degrees means it's directly upwind relative to the target volcano.
DISTANCES_GRID = [25, 50, 100, 150, 200] # km away from target
ANGLES_GRID = [0, 45, 90, 135, 180] # Degrees relative to wind (0 = directly downwind, 90 = crosswind, 180 = directly upwind)
DU_THRESHOLDS = [0.3] # Truncation threshold; assume perfect knowledge to isolate plume effects
INTERFERING_FLUXES = [0, 0.5, 1.0, 1.5] # Varying neighbouring volcano fluxes (kton/day)

# 2. SYNTHETIC PLUME GENERATION

def calculate_plume_du(X, Y, src_x, src_y, flux_kton_day, u, Dy, k):
    """
    Evaluates Eq 6. directly outputting column amounts in DU.
    Note: A scaling factor is often needed to translate abstract kton/day 
    into realistic DU values based on the grid resolution.
    """
    # --- UNIT CONVERSIONS ---
    # Eq 6 needs mass flux in kg/s to match u (m/s) and Dy (m^2/s)
    # 1 kton = 1,000,000 kg. 1 day = 86,400 seconds.
    flux_kg_s = flux_kton_day * (1e6 / 86400.0)

    x_local = X - src_x
    y_local = Y - src_y
    
    D = np.zeros_like(X)
    mask = x_local > 0 # Plume only flows downwind
    
    x_valid = x_local[mask]
    y_valid = y_local[mask]
    
    # Physics components (Eq 6)
    coeff = (flux_kg_s / u) / np.sqrt(4 * np.pi * Dy * (x_valid / u))
    dispersion = np.exp((-u * y_valid**2) / (4 * Dy * x_valid))
    decay = np.exp((-k * x_valid) / u)
    
    # Calculate mass field in kg/m^2
    mass_field_kg_m2 = coeff * dispersion * decay
    
    # CONVERT TO DOBSON UNITS (DU)
    # 1 DU is approximately 2.857e-5 kg/m^2 for SO2
    DU_CONVERSION = 2.857e-5 
    D[mask] = mass_field_kg_m2 / DU_CONVERSION 
    
    return D

def generate_synthetic_phase1(distance_km, angle_deg, du_threshold, flux_interfere):
    """
    Replaces your run_phase1_extraction. Generates the plumes, integrates them,
    and saves the CSVs required by the Flux Calculator tool.
    """
    # 1. Setup 2D Grid (in meters, roughly matching 0.05 deg resolution)
    # Using 5000m spacing to match A = 25km^2
    x_coords = np.arange(-200e3, 800e3, 5000) 
    y_coords = np.arange(-500e3, 500e3, 5000)
    X, Y = np.meshgrid(x_coords, y_coords)
    
    # Calculate distance from origin (Target Volcano) for every pixel in km
    grid_dist_km = np.sqrt(X**2 + Y**2) / 1000.0

    # 2. Atmospheric & Source Parameters
    u = 5.0        # Wind speed m/s (Blows along positive X axis)
    Dy = 1e4         # Cross-plume diffusivity m^2/s
    k = 1e-5         # Gas loss rate 1/s
    sigma_CA = 0.3   # Background noise in DU
    
    flux_target = 0.0   # Target volcano flux (kton/day)

    # 3. Position Interfering Volcano using polar coordinates
    distance_m = distance_km * 1000.0
    theta_rad = np.radians(angle_deg)
    src2_x = distance_m * np.cos(theta_rad)
    src2_y = distance_m * np.sin(theta_rad)

    # 4. Generate Plumes + Noise
    target_plume = calculate_plume_du(X, Y, 0, 0, flux_target, u, Dy, k)
    neighbour_plume = calculate_plume_du(X, Y, src2_x, src2_y, flux_interfere, u, Dy, k)
    
    noise = np.random.normal(0, sigma_CA, X.shape)
    total_du = target_plume + neighbour_plume + noise

    # 5. Apply Truncation (Your 'grid_so2_final' logic)
    total_du[total_du < du_threshold] = 0.0

    # 6. Spatial Integration (Mimicking your for-loop over CONFIG['radii'])
    so2_results = []
    cf_results = []
    
    # Simulate 3 consecutive days to satisfy any time-series array requirements in the Flux Calculator tool
    sim_times = [
        "2023-01-01 12:00:00",
        "2023-01-02 12:00:00",
        "2023-01-03 12:00:00"
    ]

    for sim_time in sim_times:
        for r in RADII:
            # Mask pixels within radius
            in_circle = grid_dist_km <= r
            pixels_in_circle = total_du[in_circle]
            
            so2_sum = (pixels_in_circle * KAPPA).sum()
            npoints = (pixels_in_circle > 0).sum()
            
            # Compile row identical to real script
            metrics = {
                "Time": sim_time,
                "TROPOMI_SO2-7km_mass": so2_sum,
                "radius": r,
                "npoints": npoints,
                "proportion_missing": 0.0, # Perfect synthetic grid
                "centroid_lon": 110.446,   # Dummy coordinates
                "centroid_lat": -7.54,
                "fraction_filtered_edge_tracks": 0.0,
                "fraction_filtered_quality": 0.0,
                "fraction_filtered_sza": 0.0,
                "fraction_filtered_total": 0.0
            }
            so2_results.append(metrics)
            
            # Dummy cloud data as the Flux Calculator tool requires this file to exist for processing
            cf_metrics = metrics.copy()
            del cf_metrics["TROPOMI_SO2-7km_mass"]
            cf_metrics["TROPOMI_CloudFraction_mean"] = 0.0 
            cf_results.append(cf_metrics)

    # 7. Save to CSVs WITH the Metadata Header
    rad_str = "-".join(map(str, sorted(RADII))) + "km"
    label = f"Dist{distance_km}_Ang{angle_deg}_DU{du_threshold}_IntFlux{flux_interfere}"
    
    fn_so2 = os.path.join(OUTPUT_DIR, f"TROPOMI_SO2-7km_mass_{rad_str}_Synthetic_{label}.csv")
    fn_cf = os.path.join(OUTPUT_DIR, f"TROPOMI_CloudFraction_mean_{rad_str}_Synthetic_{label}.csv")
    
    # Replicate the metadata header from your Phase 1 script
    dummy_meta = {
        "volcano": "Synthetic_Merapi",
        "latitude": -7.54, "longitude": 110.446,
        "radii": RADII,
        "filters": {"tracks": 0, "du": du_threshold, "qa": 0, "sza": 70},
        "start": "2023-01-01", "end": "2023-01-03",
        "grid_res": 0.05, "max_gap_deg": 1.0 
    }
    
    with open(fn_so2, 'w', newline='') as f:
        so2_meta = {"variable": "TROPOMI_SO2-7km_mass", **dummy_meta}
        f.write(f"# Computation info: {json.dumps(so2_meta)}\n")
        pd.DataFrame(so2_results).to_csv(f, index=False, lineterminator='\n')

    with open(fn_cf, 'w', newline='') as f:
        cf_meta = {"variable": "TROPOMI_CloudFraction_mean", **dummy_meta}
        f.write(f"# Computation info: {json.dumps(cf_meta)}\n")
        pd.DataFrame(cf_results).to_csv(f, index=False, lineterminator='\n')
        
    return fn_so2, fn_cf

# 3. PHASE 2 INVERSION (Flux Calculator TOOL)

def run_phase2_inversion(so2_csv, cf_csv, label):
    """
    Directly runs the Flux Calculator tool on the synthetic CSVs.
    """
    output_filename = f"Synthetic_Flux_{label}"
    
    # Use the same constant wind speed here to match the simulation 'u = 5.0'
    command = [
        "so2_flux_calculator",
        so2_csv.replace("\\", "/"),
        cf_csv.replace("\\", "/"),
        "--method", "curvefit",
        "--wind_speed", "5",
        "--output_file", output_filename,
        "--output_directory", TS_DIR.replace("\\", "/")
    ]
    
    try:
        # We pass check=False so it doesn't halt the whole script, but we capture the error
        result = subprocess.run(command, capture_output=True, text=True, shell=True)
        
        # If the Flux Calculator tool crashed, print exactly why it crashed
        if result.returncode != 0:
            print(f"\n[!] The Flux Calculator tool crashed on {label} [!]")
            print(result.stderr)
            print("-" * 40 + "\n")
            
    except Exception as e:
        print(f"Subprocess failed to launch: {e}")
        
    return os.path.join(TS_DIR, output_filename + ".csv")

# 4. EXECUTION LOOP

if __name__ == '__main__':
    all_combinations = list(itertools.product(DISTANCES_GRID, ANGLES_GRID, DU_THRESHOLDS, INTERFERING_FLUXES))
    print(f"Starting Simulation of {len(all_combinations)} scenarios...")
    
    results_log = []
    true_flux = 0.0 # The flux_target we set in the generation function
    
    for dist, ang, du, int_flux in all_combinations:
        label = f"Dist{dist}_Ang{ang}_DU{du}_IntFlux{int_flux}"
        print(f"Processing: {label}")
        
        # 1. Generate Synthetic Grid and Integrate
        so2_path, cf_path = generate_synthetic_phase1(dist, ang, du, int_flux)
        
        # 2. Run CLI Inversion
        ts_path = run_phase2_inversion(so2_path, cf_path, label)
        
        # 3. Evaluate Performance
        try:
            tropomi_df = pd.read_csv(ts_path, comment='#')
            # Extract the retrieved flux and uncertainty (1 row since only 1 day was simulated)
            retrieved_flux = tropomi_df['flux'].iloc[0]
            uncertainty = tropomi_df['sigma_flux'].iloc[0]
            
            error = retrieved_flux - true_flux
            
            results_log.append({
                'Distance_km': dist,
                'Angle_deg': ang,
                'DU_Threshold': du,
                'Interfering_Flux': int_flux,
                'True_Flux': true_flux,
                'Retrieved_Flux': retrieved_flux,
                'Error': error,
                'Reported_Uncertainty': uncertainty
            })
        except Exception as e:
            print(f"Could not read output for {label}: {e}")

    # Save final sensitivity analysis
    df_results = pd.DataFrame(results_log)
    df_results.to_csv(os.path.join(TS_DIR, "Synthetic_Sensitivity_Results_Neighbour_Degas.csv"), index=False)
    print("\nSimulation Complete. Results saved.")

In [ ]:
## Figure 6: Heatmap visualisation of Sensitivity Analysis 
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Paths
# ------------------------------------------------------------------------ INSERT PATH FOR FOLDERS HERE -------------------------------------------------------------------------
TS_DIR = r"XXX\synthetic\XXX" # Replace with the path to folder where files from either scenarios are stored. Run one scenario at a time
RESULTS_FILE = os.path.join(TS_DIR, "Synthetic_Sensitivity_Results_Neighbour_Degas.csv")
OUTPUT_IMAGE = os.path.join(TS_DIR, r"XXX.png")

def plot_pollution_heatmap():
    sns.set_context("talk", font_scale=1)

    df = pd.read_csv(RESULTS_FILE)
    int_fluxes = sorted(df['Interfering_Flux'].unique())
    
    # Setup figure with dedicated colorbar column
    fig, axes = plt.subplots(1, len(int_fluxes) + 1, 
                             figsize=(22, 5), 
                             gridspec_kw={'width_ratios': [1]*len(int_fluxes) + [0.08]})
    
    cbar_ax = axes[-1]
    plot_axes = axes[:-1]
    
    angle_labels = {0: '0°\n(Downwind)', 90: '90°\n(Crosswind)', 180: '180°\n(Upwind)'}

    for i, (ax, int_flux) in enumerate(zip(plot_axes, int_fluxes)):
        subset = df[df['Interfering_Flux'] == int_flux]
        heatmap_data = subset.pivot(index="Angle_deg", columns="Distance_km", values="Retrieved_Flux")
        heatmap_data = heatmap_data.sort_index(ascending=True)
        heatmap_data.index = [angle_labels.get(val, f"{val}°") for val in heatmap_data.index]
        
        sns.heatmap(
            heatmap_data, 
            ax=ax, 
            cmap="coolwarm", 
            center=0.0, # CHANGE ACCORDINGLY to 0 or 1 depending on whether target is degassing
            annot=True, 
            fmt=".2f", 
            vmin=0.0, 
            vmax=2.5, 
            cbar=(i == 0), 
            cbar_ax=cbar_ax, 
            cbar_kws={'label': 'Retrieved Flux (kton/day)'}
        )
        
        # Single-line title looks cleaner
        ax.set_title(f'Neighbour: {int_flux} kton/day', fontweight='bold', pad=15)
        ax.set_xlabel('Distance (km)', fontweight='bold')
        
        # --- THE FIX FOR THE UGLY Y-AXES ---
        if i == 0:
            # First plot: Show labels, force them to be horizontal (rotation=0)
            ax.set_ylabel('Angle Relative to Target', fontweight='bold')
            ax.tick_params(axis='y', rotation=0) 
        else:
            # All other plots: completely hide the Y-axis title and tick labels
            ax.set_ylabel('') 
            ax.set_yticks([]) 

    plt.suptitle('Sensitivity of SO$_2$ Flux Retrieval to Interfering Volcanoes (Target = 0 kton/day)', 
                 fontsize=24, fontweight='bold', y=1.05)
                 
    # Use subplots_adjust instead of tight_layout for finer control
    # wspace=0.1 brings the heatmaps closer together since we removed the redundant text
    plt.subplots_adjust(top=0.85, bottom=0.15, wspace=0.1)
    
    plt.savefig(OUTPUT_IMAGE, dpi=300, bbox_inches='tight', transparent=True)
    print(f"Heatmap saved to: {OUTPUT_IMAGE}")
    plt.show()

if __name__ == '__main__':
    plot_pollution_heatmap()

In [ ]:
## Figure 7 (Part 1): Visualising the simulated plumes on a map for a Single Scenario (Run seperately for each scenario)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Plume Calculation (same as before)
def calculate_plume_du(X, Y, src_x, src_y, flux_kton_day, u, Dy, k):
    flux_kg_s = flux_kton_day * (1e6 / 86400.0)
    x_local = X - src_x
    y_local = Y - src_y
    
    D = np.zeros_like(X)
    mask = x_local > 0 
    
    x_valid = x_local[mask]
    y_valid = y_local[mask]
    
    coeff = (flux_kg_s / u) / np.sqrt(4 * np.pi * Dy * (x_valid / u))
    dispersion = np.exp((-u * y_valid**2) / (4 * Dy * x_valid))
    decay = np.exp((-k * x_valid) / u)
    
    mass_field_kg_m2 = coeff * dispersion * decay
    DU_CONVERSION = 2.857e-5 
    D[mask] = mass_field_kg_m2 / DU_CONVERSION 
    
    return D

# ------------------------------------------------------------------------ 2. Configuration -----------------------------------------------------------------------------------------
# Change these to visualize different setups!
TARGET_FLUX = 1.0       # Target volcano flux (kton/day)
NEIGHBOUR_FLUX = 1.5    # Neighboring volcano flux
DISTANCE_KM = 200.0      # Distance to neighbor (km)
ANGLE_DEG = 90         # 0=Downwind, 90=Crosswind, 180=Upwind
DU_THRESHOLD = 0.3      # Truncation threshold

# 3. Generate Map
# Setup Grid (in meters)
x_coords = np.arange(-200e3, 800e3, 5000)
y_coords = np.arange(-500e3, 500e3, 5000)
X, Y = np.meshgrid(x_coords, y_coords)

# Atmospheric Parameters
u, Dy, k, sigma_CA = 5.0, 1e4, 1e-5, 0.3

# Position Neighbor
dist_m = DISTANCE_KM * 1000.0
theta_rad = np.radians(ANGLE_DEG)
src2_x = dist_m * np.cos(theta_rad)
src2_y = dist_m * np.sin(theta_rad)

# Generate Plumes + Noise
target_plume = calculate_plume_du(X, Y, 0, 0, TARGET_FLUX, u, Dy, k)
neighbor_plume = calculate_plume_du(X, Y, src2_x, src2_y, NEIGHBOUR_FLUX, u, Dy, k)
noise = np.random.normal(0, sigma_CA, X.shape)

total_du = target_plume + neighbor_plume + noise

# Apply Truncation (Set below-threshold pixels to NaN so they show as white background)
total_du[total_du < DU_THRESHOLD] = np.nan

# 4. Plotting
fig, ax = plt.subplots(figsize=(12, 8))

# Define the physical extent of the image in kilometers for the axes
extents_km = [x_coords.min()/1000, x_coords.max()/1000, y_coords.min()/1000, y_coords.max()/1000]

# Plot the SO2 Heatmap
im = ax.imshow(total_du, extent=extents_km, origin='lower', cmap='viridis', 
               vmin=DU_THRESHOLD, vmax=np.nanmax(total_du)*0.6) # Capped vmax to highlight the tail

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('SO$_2$ Column Mass', fontsize=14, fontweight='bold')
cbar.ax.tick_params(labelsize=12)

# Plot Volcano Locations
ax.plot(0, 0, '^', color='black', markersize=14, markeredgecolor='white', label='Target Volcano')
ax.plot(src2_x/1000, src2_y/1000, '^', color='red', markersize=14, markeredgecolor='white', label=f'Neighbour Volcano')

# Overlay Integration Radii
radii_to_draw = [100, 200, 300, 400, 500] # in km
colors = ['black'] * len(radii_to_draw) # All circles in black for visibility

for i, (r, c) in enumerate(zip(radii_to_draw, colors)):
    # Only assign the text label to the very first circle
    if i == 0:
        legend_label = 'Radii from 100-500km (100km intervals)'
    else:
        # Tell matplotlib to ignore the rest of the circles in the legend
        legend_label = "_nolegend_" 
        
    circle = patches.Circle((0, 0), r, fill=False, edgecolor=c, linestyle='--', linewidth=2, alpha=0.8, label=legend_label)
    ax.add_patch(circle)

# Formatting
ax.set_xlabel('East-West Distance (km)', fontweight='bold', fontsize=14)
ax.set_ylabel('North-South Distance (km)', fontweight='bold', fontsize=14)
ax.tick_params(axis='both', which='major', labelsize=14)

# Fix axis limits to focus on the interesting part
ax.set_xlim(-200, 600)
ax.set_ylim(-300, 300)

ax.legend(loc='upper right', framealpha=0.9, fontsize=12)
ax.grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r'XXX.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
## Figure 7 (Part 2): Curve Fitting Deconstruction for a Single Scenario (Run seperately for each scenario)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Paths and Files ---
# --------------------------------------------------------------------- INSERT PATH FOR SIMULATION FOLDERS HERE ----------------------------------------------------------------------
OUTPUT_DIR = r"XXX\synthetic\both"
TS_DIR = r"XXX\synthetic\both"

# Choose ONE scenario to decompose (e.g., the 0-degree Downwind trap)
MASS_FILE = os.path.join(OUTPUT_DIR, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Synthetic_Dist200_Ang0_DU0.3_IntFlux1.5.csv")
FLUX_FILE = os.path.join(TS_DIR, "Synthetic_Flux_Dist200_Ang0_DU0.3_IntFlux1.5.csv")

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
OUTPUT_IMAGE = os.path.join(r"XXX", "Dist200_Ang0_DU0.3_IntFlux1.5.svg")

def plot_deconstructed_single_fit():
    # --- 2. Load Data ---
    df_mass = pd.read_csv(MASS_FILE, comment='#')
    df_flux = pd.read_csv(FLUX_FILE, comment='#')

    # Filter for a single day and sort by radius
    target_time = df_mass['Time'].unique()[0]
    d_mass = df_mass[df_mass['Time'] == target_time].sort_values('radius')

    # Extract the curve parameters calculated by the tool
    a0 = df_flux['a'].iloc[0]
    a1 = df_flux['b'].iloc[0]
    a2 = df_flux['c'].iloc[0]
    retrieved_flux = df_flux['flux'].iloc[0]

    # --- 3. Generate Smooth Mathematical Curves ---
    r_smooth = np.linspace(0, 500, 100)
    
    # The individual components
    linear_comp = a1 * r_smooth
    quadratic_comp = a2 * (r_smooth**2)
    
    # The combined fit
    combined_fit = a0 + linear_comp + quadratic_comp

    # --- 4. Plotting Setup ---
    sns.set_theme(style="whitegrid", context="talk")
    fig, ax = plt.subplots(figsize=(12, 8))

    # 1. The Linear Component (This determines the Flux!)
    ax.plot(r_smooth, linear_comp, linestyle=':', color='#27ae60', linewidth=3, 
            label=f'Linear Component ($a$)')
            
    # 2. The Quadratic Component (This models background/noise)
    ax.plot(r_smooth, quadratic_comp, linestyle='-.', color="#8e44ad", linewidth=3, 
            label=f'Quadratic Component ($b$)')

    # 3. Combined Fit
    ax.plot(r_smooth, combined_fit, linestyle='-', color='#c0392b', linewidth=3.5, 
            label=f'Combined Fit\n($M = \epsilon_n + a r + b r^2$)')
            
    # 4. Raw Synthetic Data
    ax.plot(d_mass['radius'], d_mass['TROPOMI_SO2-7km_mass'], 
            marker='o', markersize=9, linewidth=0, color='black', 
            label='Raw Synthetic Data')

    # --- 5. Formatting ---
    ax.set_xlabel('Integration Radius (km)', fontweight='bold')
    ax.set_ylabel('Mass Contribution (ktons)', fontweight='bold')
    ax.set_title(f'Grandin Algorithm Inversion\n(Retrieved Flux: {retrieved_flux:.2f} kt/d | Target: 1.0 kt/d | Neighbour: 1.5 kt/d at 200km Downwind)', 
                 fontweight='bold', fontsize=20, pad=15)

    ax.set_xlim(0, 520)
    
    # Optional: Highlight the zero-line to show where the quadratic term pulls things negative
    ax.axhline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)

    ax.legend(loc='best', fontsize='medium', framealpha=0.9, shadow=True)
    ax.grid(True, linestyle=':', alpha=0.7)

    plt.tight_layout()
    plt.savefig(OUTPUT_IMAGE, dpi=300, bbox_inches='tight')
    print(f"Single Deconstruction plot saved to:\n{OUTPUT_IMAGE}")
    plt.show()

if __name__ == '__main__':
    plot_deconstructed_single_fit()

In [ ]:
## Appendix Figure A-1: Plotting two Raw TROPOMI swaths together to visualise swath gap
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import cartopy.crs as ccrs 
import cartopy.feature as cfeature
from geopy.distance import distance
import os

# 1. Open both datasets
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-1")  # Navigate to folder with the data files
path_east = os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230612T044438_20230612T062608_29337_03_020401_20230614T023007.nc")
path_west = os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230612T062608_20230612T080738_29338_03_020401_20230614T034050.nc")    

# --- Open PRODUCT group ---
ds_e = xr.open_dataset(path_east, group="PRODUCT").set_coords(['longitude', 'latitude'])
ds_w = xr.open_dataset(path_west, group="PRODUCT").set_coords(['longitude', 'latitude'])

# --- Open GEOLOCATIONS group (FIX for KeyError) ---
geo_e = xr.open_dataset(path_east, group="PRODUCT/SUPPORT_DATA/GEOLOCATIONS")
geo_w = xr.open_dataset(path_west, group="PRODUCT/SUPPORT_DATA/GEOLOCATIONS")

# 2. Define the geographic slice
lat_min, lat_max = -13.0, -2.0  # Slightly expanded to fit the 500km circle comfortably
lon_min, lon_max = 105.0, 124.0

def get_sliced_so2(ds, geo, sza_max=70):
    sza = geo['solar_zenith_angle']

    mask = (
        (ds.latitude > lat_min) & (ds.latitude < lat_max) &
        (ds.longitude > lon_min) & (ds.longitude < lon_max) &
        (ds.qa_value > 0.1) &
        (sza < sza_max)   # SZA filter added
    )

    sliced = ds['sulfurdioxide_total_vertical_column'].where(mask, drop=True)
    return sliced if sliced.size > 0 else None

so2_e_sliced = get_sliced_so2(ds_e, geo_e)
so2_w_sliced = get_sliced_so2(ds_w, geo_w)

# 3. Setup Plot with Cartopy Projection
fig = plt.figure(figsize=(12, 10))
# Using PlateCarree projection for the axes
ax = plt.axes(projection=ccrs.PlateCarree())

# Add map features for context
ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)

# Define consistent color limits
vmin, vmax = -0.0005, 0.0015 

# 4. Plot both swaths
# transform=ccrs.PlateCarree() tells Cartopy our data is in degrees
im = None
if so2_w_sliced is not None:
    im = so2_w_sliced.plot(ax=ax, x='longitude', y='latitude', transform=ccrs.PlateCarree(),
                           vmin=vmin, vmax=vmax, cmap='viridis', add_colorbar=False)

if so2_e_sliced is not None:
    im_e = so2_e_sliced.plot(ax=ax, x='longitude', y='latitude', transform=ccrs.PlateCarree(),
                             vmin=vmin, vmax=vmax, cmap='viridis', add_colorbar=False)
    if im is None: im = im_e

# Add a colorbar
if im:
    plt.colorbar(im, ax=ax, label='SO2 Vertical Column [mol m-2]', extend='both', 
                 orientation='horizontal', pad=0.08)

# 5. Add Volcano Marker and 500km Radius
v_lat, v_lon = -7.54, 110.446
ax.plot(v_lon, v_lat, 'ro', markersize=8, transform=ccrs.PlateCarree(), label='Mt. Merapi')

def get_circle_points(lat, lon, radius_km):
    pts = []
    for bearing in range(0, 365, 5):
        dest = distance(kilometers=radius_km).destination((lat, lon), bearing)
        pts.append([dest.longitude, dest.latitude])
    return np.array(pts)

circle_pts = get_circle_points(v_lat, v_lon, 500)
ax.plot(circle_pts[:, 0], circle_pts[:, 1], 'r--', linewidth=2, 
        transform=ccrs.PlateCarree(), label='500km Radius')

# 6. Formatting
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
plt.title("Raw Image (12th June 2023)", fontsize=14)
plt.legend(loc='upper right')

# Add gridlines with labels
gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
gl.top_labels = False
gl.right_labels = False
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
## Combined Mass Integration and Cloud Fraction Mean Calculation (Truncation Before Interpolation)

import os, json, numpy as np, pandas as pd
from netCDF4 import Dataset
from glob import glob
from collections import defaultdict
from datetime import datetime
from scipy.spatial import ConvexHull, cKDTree
from scipy.interpolate import griddata
from matplotlib.path import Path

# --- CONFIG & PATHS ---
DOWNLOAD_DIR = r"C:\Users\Hao Wei\Documents\FYP\RS Data\downloads"
OUTPUT_DIR = r"C:\Users\Hao Wei\Documents\FYP\RS Data\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Master Config
CONFIG = {
    "volcano": "Merapi",
    "latitude": -7.54, "longitude": 110.446,
    "radii": [500, 400, 300, 250, 200, 150, 100, 75, 50, 25],
    "filters": {"tracks": 7, "du": 0, "qa": 0, "sza": 70},
    "start": "2023-06-01", "end": "2023-12-30",
    "grid_res": 0.05, "max_gap_deg": 1.0
}

# Grandin's exact Kappa for assumed Area A = 25km2
KAPPA = 0.0007

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2, dp, dl = np.radians(lat1), np.radians(lat2), np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def get_safe_overhead_time(p_group, t_lat, t_lon, fname):
    try:
        dt_str = fname.split('____')[1][:15]
        dt = datetime.strptime(dt_str, "%Y%m%dT%H%M%S")
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception as e:
        print(f"Warning: Could not parse time from filename {fname}. Error: {e}")
        return "Unknown Time"

def extract_nc_combined(path, t_lat, t_lon):
    with Dataset(path, 'r') as root:
        p = root.groups['PRODUCT']
        res = p.groups['SUPPORT_DATA'].groups['DETAILED_RESULTS']
        input_data = p.groups['SUPPORT_DATA'].groups['INPUT_DATA']
        geo = p.groups['SUPPORT_DATA'].groups['GEOLOCATIONS']
        
        v_so2_raw = res.variables['sulfurdioxide_total_vertical_column_7km']
        conv = v_so2_raw.getncattr('multiplication_factor_to_convert_to_DU')
        cf_raw = input_data.variables['cloud_fraction_crb']
        
        lat_full, lon_full = p.variables['latitude'][:], p.variables['longitude'][:]
        crop = (lat_full > t_lat - 6) & (lat_full < t_lat + 6) & (lon_full > t_lon - 6) & (lon_full < t_lon + 6)
        if not np.any(crop): return None, None
        
        so2 = np.ma.filled(v_so2_raw[:].astype(float), np.nan)[crop] * conv
        cf = np.ma.filled(cf_raw[:].astype(float), np.nan)[crop]
        
        df = pd.DataFrame({
            'lat': lat_full[crop], 'lon': lon_full[crop],
            'qa': p.variables['qa_value'][:][crop],
            'sza': geo.variables['solar_zenith_angle'][:][crop],
            'row': np.tile(np.arange(p.dimensions['ground_pixel'].size), p.dimensions['scanline'].size).reshape(lat_full.shape)[crop],
            'so2': so2, 'cf': cf
        })
        return df, get_safe_overhead_time(p, t_lat, t_lon, os.path.basename(path))

# --- EXECUTION ---
files = glob(os.path.join(DOWNLOAD_DIR, "*.nc"))
start_dt, end_dt = datetime.strptime(CONFIG['start'], "%Y-%m-%d"), datetime.strptime(CONFIG['end'], "%Y-%m-%d")
groups = defaultdict(list)
for f in files:
    d_str = os.path.basename(f).split('____')[1][:8]
    if start_dt <= datetime.strptime(d_str, "%Y%m%d") <= end_dt:
        groups[d_str].append(f)

so2_results, cf_results = [], []

for date_key in sorted(groups.keys()):
    paths = groups[date_key]
    paths.sort(key=lambda x: int(os.path.basename(x).split('_')[12]))
    
    daily_data, final_time = [], ""
    for p_path in paths:
        df, t = extract_nc_combined(p_path, CONFIG['latitude'], CONFIG['longitude'])
        if df is not None:
            daily_data.append(df); final_time = t

    if not daily_data: continue
    master = pd.concat(daily_data, ignore_index=True)
    master['dist'] = haversine(CONFIG['latitude'], CONFIG['longitude'], master['lat'], master['lon'])
    
    m_e = (master['row'] < CONFIG['filters']['tracks']) | (master['row'] > (449 - CONFIG['filters']['tracks']))
    m_q, m_s = (master['qa'] < CONFIG['filters']['qa']), (master['sza'] > CONFIG['filters']['sza'])
    
    soundings = master[~(m_e | m_q | m_s | master['so2'].isna())].copy()
    if len(soundings) < 4: continue

    lons = np.arange(CONFIG['longitude']-5.5, CONFIG['longitude']+5.5, CONFIG['grid_res'])
    lats = np.arange(CONFIG['latitude']-5.5, CONFIG['latitude']+5.5, CONFIG['grid_res'])
    grid_lon, grid_lat = np.meshgrid(lons, lats)
    grid_pts = np.column_stack([grid_lon.ravel(), grid_lat.ravel()])
    
    pts = soundings[['lon', 'lat']].values
    hull_path = Path(pts[ConvexHull(pts).vertices])
    
    # --- DU THRESHOLDING BEFORE INTERPOLATION ---
    # Apply the DU filter directly to the raw satellite soundings.
    # Values below the threshold become 0.0 so they act as zero-anchors during griddata interpolation.
    raw_so2_filtered = np.where(soundings['so2'].values >= CONFIG['filters']['du'], soundings['so2'].values, 0.0)
    
    grid_so2 = griddata(pts, raw_so2_filtered, grid_pts, method='linear')
    # -----------------------------------------------------
    
    grid_cf = griddata(pts, soundings['cf'].values, grid_pts, method='linear')
    dists, _ = cKDTree(pts).query(grid_pts)
    
    mask = hull_path.contains_points(grid_pts) & (dists <= CONFIG['max_gap_deg'])
    
    # Apply the spatial mask (region outside Convex Hull and max gap to exactly 0.0)
    grid_so2_final = np.where(mask, np.nan_to_num(grid_so2, nan=0.0), 0.0)
    
    grid_cf_final = np.where(mask, np.nan_to_num(grid_cf, nan=0.0), np.nan)
    
    g_df = pd.DataFrame({'lon': grid_pts[:,0], 'lat': grid_pts[:,1], 'so2': grid_so2_final, 'cf': grid_cf_final})
    g_df['dist'] = haversine(CONFIG['latitude'], CONFIG['longitude'], g_df['lat'], g_df['lon'])

    for r in CONFIG['radii']:
        r_orig = master[master['dist'] <= r]
        if r_orig.empty: continue
        
        f_e = (r_orig['row'] < CONFIG['filters']['tracks']) | (r_orig['row'] > (449 - CONFIG['filters']['tracks']))
        f_q, f_s = r_orig['qa'] < CONFIG['filters']['qa'], r_orig['sza'] > CONFIG['filters']['sza']

        r_grid = g_df[g_df['dist'] <= r]
        
        # Coverage Validation Logic
        total_grid_points = len(r_grid) 
        
        if total_grid_points > 0:
            r_soundings = soundings[soundings['dist'] <= r]
            
            if r_soundings.empty:
                proportion_missing = 100.0
            else:
                valid_tree = cKDTree(r_soundings[['lon', 'lat']].values)
                grid_coords = r_grid[['lon', 'lat']].values
                dists_to_valid, _ = valid_tree.query(grid_coords)
                
                valid_grid_points = (dists_to_valid <= CONFIG['grid_res']).sum()
                proportion_missing = (total_grid_points - valid_grid_points) / total_grid_points * 100
        else:
            proportion_missing = np.nan

        so2_sum = (r_grid['so2'] * KAPPA).sum()
        npoints = (r_grid['so2'] > 0).sum()
        
        if not r_grid.empty and r_grid['so2'].sum() > 0:
            c_lat = np.average(r_grid['lat'], weights=r_grid['so2'])
            c_lon = np.average(r_grid['lon'], weights=r_grid['so2'])
            cf_mean = r_grid['cf'].dropna().mean()
        else:
            c_lat, c_lon, cf_mean = CONFIG['latitude'], CONFIG['longitude'], r_grid['cf'].dropna().mean() if not r_grid['cf'].dropna().empty else 0.0

        other_metrics = {
            "radius": r, "npoints": npoints,
            "proportion_missing": proportion_missing, 
            "centroid_lon": c_lon, "centroid_lat": c_lat,
            "fraction_filtered_edge_tracks": f_e.mean()*100,
            "fraction_filtered_quality": f_q.mean()*100,
            "fraction_filtered_sza": f_s.mean()*100,
            "fraction_filtered_total": (f_e | f_q | f_s).mean()*100
        }
        
        so2_results.append({"Time": final_time, "TROPOMI_SO2-7km_mass": so2_sum, **other_metrics})
        cf_results.append({"Time": final_time, "TROPOMI_CloudFraction_mean": cf_mean, **other_metrics})

    print(f"Processed: {date_key}")

# --- DYNAMIC FILENAMES ---
rad_str = "-".join(map(str, sorted(CONFIG['radii']))) + "km"

fn_so2 = (f"TROPOMI_SO2-7km_mass_{rad_str}_{CONFIG['volcano']}_"
          f"tracks-{CONFIG['filters']['tracks']}_du-{CONFIG['filters']['du']}_"
          f"qa-{CONFIG['filters']['qa']}_sza-{CONFIG['filters']['sza']}_"
          f"{CONFIG['start']}_{CONFIG['end']}.csv")

fn_cf = (f"TROPOMI_CloudFraction_mean_{rad_str}_{CONFIG['volcano']}_"
         f"tracks-{CONFIG['filters']['tracks']}_"
         f"qa-{CONFIG['filters']['qa']}_sza-{CONFIG['filters']['sza']}_"
         f"{CONFIG['start']}_{CONFIG['end']}.csv")

# --- SAVE WITH METADATA ---
with open(os.path.join(OUTPUT_DIR, fn_so2), 'w', newline='') as f:
    so2_meta = CONFIG.copy()
    so2_meta = {"variable": "TROPOMI_SO2-7km_mass", **so2_meta}
    f.write(f"# Computation info: {json.dumps(so2_meta)}\n")
    pd.DataFrame(so2_results).to_csv(f, index=False, lineterminator='\n')

with open(os.path.join(OUTPUT_DIR, fn_cf), 'w', newline='') as f:
    cf_meta = CONFIG.copy()
    cf_meta = {"variable": "TROPOMI_CloudFraction_mean", **cf_meta}
    f.write(f"# Computation info: {json.dumps(cf_meta)}\n")
    pd.DataFrame(cf_results).to_csv(f, index=False, lineterminator='\n')

print(f"Done! Reports saved to {OUTPUT_DIR}")

In [ ]:
## Appendix Figure A-2 (Part 1): Comparing the differences in SO2 mass between my recreated data and Grandin et al.'s flux calculator data when truncation is applied before interpolation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def load_and_sort(filepath):
    df = pd.read_csv(filepath, comment='#')
    df['Time'] = pd.to_datetime(df['Time']).dt.normalize()
    df = df.sort_values(by=['Time', 'radius']).reset_index(drop=True)
    return df

# 1. Load and Sort
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-2")  # Navigate to folder with the data files
path_mine = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-7_du-0_qa-0_sza-70_2023-06-01_2023-08-30.csv")
path_authors = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-7_du-0_qa-0_sza-70_2023-06-01_2023-08-16.csv")    

df_mine = load_and_sort(path_mine)
df_authors = load_and_sort(path_authors)

# 2. Synchronize
df_mine = df_mine[df_mine['Time'] <= df_authors['Time'].max()].reset_index(drop=True)
df_authors = df_authors[df_authors['Time'] >= df_mine['Time'].min()].reset_index(drop=True)

# 3. Subtraction (Only calculating the SO2 mass difference now)
df_diff = df_mine.copy()
df_diff['TROPOMI_SO2-7km_mass'] = df_mine['TROPOMI_SO2-7km_mass'] - df_authors['TROPOMI_SO2-7km_mass']

# 4. Plotting
fig, ax = plt.subplots(figsize=(14, 8)) # Adjusted height for a single plot
sns.set_style("whitegrid")

radii_order = [25, 50, 75, 100, 150, 200, 250, 300, 400, 500]
# Using 'viridis' for clear color differentiation
palette = sns.color_palette("viridis", n_colors=len(radii_order))

# Plotting just the SO2 mass
sns.lineplot(
    data=df_diff, 
    x='Time', 
    y='TROPOMI_SO2-7km_mass', 
    hue='radius', 
    palette=palette, 
    ax=ax, 
    marker='o',
    markersize=6,
    hue_order=radii_order,
    legend='full' # Force full legend data generation
)

ax.axhline(0, color='black', linestyle='--', alpha=0.8)
ax.set_ylabel('Difference in SO2 Mass (kt)', fontsize=20)
ax.set_xlabel('Date', fontsize=20)
ax.set_title('A. Comparison of Integrated SO2 Mass (Recreated - VolcPlume): Truncation before Interpolation', fontweight='bold', fontsize=22)

# Force the legend to show all unique labels
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, 
    labels=labels, 
    title='Radius (km)', 
    title_fontsize=18, 
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    borderaxespad=0
)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
## Combined Mass Integration and Cloud Fraction Mean Calculation (Truncation After Interpolation)

import os, json, numpy as np, pandas as pd
from netCDF4 import Dataset
from glob import glob
from collections import defaultdict
from datetime import datetime
from scipy.spatial import ConvexHull, cKDTree
from scipy.interpolate import griddata
from matplotlib.path import Path

# --- CONFIG & PATHS ---
DOWNLOAD_DIR = r"C:\Users\Hao Wei\Documents\FYP\RS Data\downloads"
OUTPUT_DIR = r"C:\Users\Hao Wei\Documents\FYP\RS Data\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Master Config
CONFIG = {
    "volcano": "Merapi",
    "latitude": -7.54, "longitude": 110.446,
    "radii": [500, 400, 300, 250, 200, 150, 100, 75, 50, 25],
    "filters": {"tracks": 7, "du": 0, "qa": 0, "sza": 70},
    "start": "2023-06-01", "end": "2023-12-31",
    "grid_res": 0.05, "max_gap_deg": 1.0 
}

# Grandin's exact Kappa for assumed Area A = 25km2
KAPPA = 0.0007 

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    p1, p2, dp, dl = np.radians(lat1), np.radians(lat2), np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def get_safe_overhead_time(p_group, t_lat, t_lon, fname):
    try:
        dt_str = fname.split('____')[1][:15]
        dt = datetime.strptime(dt_str, "%Y%m%dT%H%M%S")
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception as e:
        print(f"Warning: Could not parse time from filename {fname}. Error: {e}")
        return "Unknown Time"

def extract_nc_combined(path, t_lat, t_lon):
    with Dataset(path, 'r') as root:
        p = root.groups['PRODUCT']
        res = p.groups['SUPPORT_DATA'].groups['DETAILED_RESULTS']
        input_data = p.groups['SUPPORT_DATA'].groups['INPUT_DATA']
        geo = p.groups['SUPPORT_DATA'].groups['GEOLOCATIONS']
        
        v_so2_raw = res.variables['sulfurdioxide_total_vertical_column_7km']
        conv = v_so2_raw.getncattr('multiplication_factor_to_convert_to_DU')
        cf_raw = input_data.variables['cloud_fraction_crb']
        
        lat_full, lon_full = p.variables['latitude'][:], p.variables['longitude'][:]
        crop = (lat_full > t_lat - 6) & (lat_full < t_lat + 6) & (lon_full > t_lon - 6) & (lon_full < t_lon + 6)
        if not np.any(crop): return None, None
        
        so2 = np.ma.filled(v_so2_raw[:].astype(float), np.nan)[crop] * conv
        cf = np.ma.filled(cf_raw[:].astype(float), np.nan)[crop]
        
        df = pd.DataFrame({
            'lat': lat_full[crop], 'lon': lon_full[crop],
            'qa': p.variables['qa_value'][:][crop],
            'sza': geo.variables['solar_zenith_angle'][:][crop],
            'row': np.tile(np.arange(p.dimensions['ground_pixel'].size), p.dimensions['scanline'].size).reshape(lat_full.shape)[crop],
            'so2': so2, 'cf': cf
        })
        return df, get_safe_overhead_time(p, t_lat, t_lon, os.path.basename(path))

# --- EXECUTION ---
files = glob(os.path.join(DOWNLOAD_DIR, "*.nc"))
start_dt, end_dt = datetime.strptime(CONFIG['start'], "%Y-%m-%d"), datetime.strptime(CONFIG['end'], "%Y-%m-%d")
groups = defaultdict(list)
for f in files:
    d_str = os.path.basename(f).split('____')[1][:8]
    if start_dt <= datetime.strptime(d_str, "%Y%m%d") <= end_dt:
        groups[d_str].append(f)

so2_results, cf_results = [], []

for date_key in sorted(groups.keys()):
    paths = groups[date_key]
    paths.sort(key=lambda x: int(os.path.basename(x).split('_')[12]))
    
    daily_data, final_time = [], ""
    for p_path in paths:
        df, t = extract_nc_combined(p_path, CONFIG['latitude'], CONFIG['longitude'])
        if df is not None:
            daily_data.append(df); final_time = t

    if not daily_data: continue
    master = pd.concat(daily_data, ignore_index=True)
    master['dist'] = haversine(CONFIG['latitude'], CONFIG['longitude'], master['lat'], master['lon'])
    
    m_e = (master['row'] < CONFIG['filters']['tracks']) | (master['row'] > (449 - CONFIG['filters']['tracks']))
    m_q, m_s = (master['qa'] < CONFIG['filters']['qa']), (master['sza'] > CONFIG['filters']['sza'])
    
    soundings = master[~(m_e | m_q | m_s | master['so2'].isna())].copy()
    if len(soundings) < 4: continue

    lons = np.arange(CONFIG['longitude']-5.5, CONFIG['longitude']+5.5, CONFIG['grid_res'])
    lats = np.arange(CONFIG['latitude']-5.5, CONFIG['latitude']+5.5, CONFIG['grid_res'])
    grid_lon, grid_lat = np.meshgrid(lons, lats)
    grid_pts = np.column_stack([grid_lon.ravel(), grid_lat.ravel()])
    
    pts = soundings[['lon', 'lat']].values
    hull_path = Path(pts[ConvexHull(pts).vertices])
    
    grid_so2 = griddata(pts, soundings['so2'].values, grid_pts, method='linear')
    grid_cf = griddata(pts, soundings['cf'].values, grid_pts, method='linear')
    dists, _ = cKDTree(pts).query(grid_pts)
    
    mask = hull_path.contains_points(grid_pts) & (dists <= CONFIG['max_gap_deg'])
    
    # DU Thresholding Logic ---
    spatially_masked_so2 = np.where(mask, np.nan_to_num(grid_so2, nan=0.0), 0.0)
    grid_so2_final = np.where(spatially_masked_so2 >= CONFIG['filters']['du'], spatially_masked_so2, 0.0)
    # ---
    
    grid_cf_final = np.where(mask, np.nan_to_num(grid_cf, nan=0.0), np.nan)
    
    g_df = pd.DataFrame({'lon': grid_pts[:,0], 'lat': grid_pts[:,1], 'so2': grid_so2_final, 'cf': grid_cf_final})
    g_df['dist'] = haversine(CONFIG['latitude'], CONFIG['longitude'], g_df['lat'], g_df['lon'])

    for r in CONFIG['radii']:
        r_orig = master[master['dist'] <= r]
        if r_orig.empty: continue
        
        f_e = (r_orig['row'] < CONFIG['filters']['tracks']) | (r_orig['row'] > (449 - CONFIG['filters']['tracks']))
        f_q, f_s = r_orig['qa'] < CONFIG['filters']['qa'], r_orig['sza'] > CONFIG['filters']['sza']

        r_grid = g_df[g_df['dist'] <= r]
        
        total_grid_points = len(r_grid)
        if total_grid_points > 0:
            r_soundings = soundings[soundings['dist'] <= r]
            
            if r_soundings.empty:
                proportion_missing = 100.0
            else:
                valid_tree = cKDTree(r_soundings[['lon', 'lat']].values)
                grid_coords = r_grid[['lon', 'lat']].values
                dists_to_valid, _ = valid_tree.query(grid_coords)
                
                valid_grid_points = (dists_to_valid <= CONFIG['grid_res']).sum()
                proportion_missing = (total_grid_points - valid_grid_points) / total_grid_points * 100
        else:
            proportion_missing = np.nan

        so2_sum = (r_grid['so2'] * KAPPA).sum()
        npoints = (r_grid['so2'] > 0).sum()
        
        if not r_grid.empty and r_grid['so2'].sum() > 0:
            c_lat = np.average(r_grid['lat'], weights=r_grid['so2'])
            c_lon = np.average(r_grid['lon'], weights=r_grid['so2'])
            cf_mean = r_grid['cf'].dropna().mean()
        else:
            c_lat, c_lon, cf_mean = CONFIG['latitude'], CONFIG['longitude'], r_grid['cf'].dropna().mean() if not r_grid['cf'].dropna().empty else 0.0

        other_metrics = {
            "radius": r, 
            "npoints": npoints,
            "proportion_missing": proportion_missing, 
            "centroid_lon": c_lon, 
            "centroid_lat": c_lat,
            "fraction_filtered_edge_tracks": f_e.mean()*100,
            "fraction_filtered_quality": f_q.mean()*100,
            "fraction_filtered_sza": f_s.mean()*100,
            "fraction_filtered_total": (f_e | f_q | f_s).mean()*100
        }
        
        so2_results.append({"Time": final_time, "TROPOMI_SO2-7km_mass": so2_sum, **other_metrics})
        cf_results.append({"Time": final_time, "TROPOMI_CloudFraction_mean": cf_mean, **other_metrics})

    print(f"Processed: {date_key}")

# --- DYNAMIC FILENAMES ---
rad_str = "-".join(map(str, sorted(CONFIG['radii']))) + "km"

fn_so2 = (f"TROPOMI_SO2-7km_mass_{rad_str}_{CONFIG['volcano']}_"
          f"tracks-{CONFIG['filters']['tracks']}_du-{CONFIG['filters']['du']}_"
          f"qa-{CONFIG['filters']['qa']}_sza-{CONFIG['filters']['sza']}_"
          f"{CONFIG['start']}_{CONFIG['end']}.csv")

fn_cf = (f"TROPOMI_CloudFraction_mean_{rad_str}_{CONFIG['volcano']}_"
         f"tracks-{CONFIG['filters']['tracks']}_"
         f"qa-{CONFIG['filters']['qa']}_sza-{CONFIG['filters']['sza']}_"
         f"{CONFIG['start']}_{CONFIG['end']}.csv")

# --- SAVE WITH METADATA ---
with open(os.path.join(OUTPUT_DIR, fn_so2), 'w', newline='') as f:
    so2_meta = CONFIG.copy()
    so2_meta = {"variable": "TROPOMI_SO2-7km_mass", **so2_meta}
    f.write(f"# Computation info: {json.dumps(so2_meta)}\n")
    pd.DataFrame(so2_results).to_csv(f, index=False, lineterminator='\n')

with open(os.path.join(OUTPUT_DIR, fn_cf), 'w', newline='') as f:
    cf_meta = CONFIG.copy()
    cf_meta = {"variable": "TROPOMI_CloudFraction_mean", **cf_meta}
    f.write(f"# Computation info: {json.dumps(cf_meta)}\n")
    pd.DataFrame(cf_results).to_csv(f, index=False, lineterminator='\n')

print(f"Done! Reports saved to {OUTPUT_DIR}")

In [ ]:
## Appendix Figure A-2 (Part 2): Comparing the differences in SO2 mass between my recreated data and Grandin et al.'s flux calculator data when truncation is applied after interpolation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def load_and_sort(filepath):
    df = pd.read_csv(filepath, comment='#')
    df['Time'] = pd.to_datetime(df['Time']).dt.normalize()
    df = df.sort_values(by=['Time', 'radius']).reset_index(drop=True)
    return df

# 1. Load and Sort
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-2")  # Navigate to folder with the data files
path_mine = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-7_du-0_qa-0_sza-70_2023-06-01_2023-12-31.csv")
path_authors = os.path.join(data_dir, "TROPOMI_SO2-7km_mass_25-50-75-100-150-200-250-300-400-500km_Merapi_tracks-7_du-0_qa-0_sza-70_2023-06-01_2023-08-16.csv")    

df_mine = load_and_sort(path_mine)
df_authors = load_and_sort(path_authors)

# 2. Synchronize
df_mine = df_mine[df_mine['Time'] <= df_authors['Time'].max()].reset_index(drop=True)
df_authors = df_authors[df_authors['Time'] >= df_mine['Time'].min()].reset_index(drop=True)

# 3. Subtraction (Only calculating the SO2 mass difference now)
df_diff = df_mine.copy()
df_diff['TROPOMI_SO2-7km_mass'] = df_mine['TROPOMI_SO2-7km_mass'] - df_authors['TROPOMI_SO2-7km_mass']

# 4. Plotting
fig, ax = plt.subplots(figsize=(14, 8)) # Adjusted height for a single plot
sns.set_style("whitegrid")

radii_order = [25, 50, 75, 100, 150, 200, 250, 300, 400, 500]
# Using 'viridis' for clear color differentiation
palette = sns.color_palette("viridis", n_colors=len(radii_order))

# Plotting just the SO2 mass
sns.lineplot(
    data=df_diff, 
    x='Time', 
    y='TROPOMI_SO2-7km_mass', 
    hue='radius', 
    palette=palette, 
    ax=ax, 
    marker='o',
    markersize=6,
    hue_order=radii_order,
    legend='full' # Force full legend data generation
)

ax.axhline(0, color='black', linestyle='--', alpha=0.8)
ax.set_ylabel('Difference in SO2 Mass (kt)', fontsize=20)
ax.set_xlabel('Date', fontsize=20)
ax.set_title('B. Comparison of Integrated SO2 Mass (Recreated - VolcPlume): Truncation after Interpolation', fontweight='bold', fontsize=22)

# Force the legend to show all unique labels
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles, 
    labels=labels, 
    title='Radius (km)', 
    title_fontsize=18,
    bbox_to_anchor=(1.02, 1), 
    loc='upper left',
    borderaxespad=0
)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.tight_layout()
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r'XXX.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
## Appendix Figure A-2 (Part 3): Comparing Flux from VolcPlume Platform and Recreated Code (Truncation before Interpolation)
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# File paths
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-2")  # Navigate to folder with the data files
flux_author = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-08-16_tracks-7_du-0_qa-0_sza-70_P550.csv")
flux_recreated = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-12-30_tracks-7_du-0_qa-0_sza-70_P550.csv") 

# Load the data, skipping comment lines
df1 = pd.read_csv(flux_author, comment='#')
df2 = pd.read_csv(flux_recreated, comment='#')

# Convert time columns to datetime
df1['time'] = pd.to_datetime(df1['time'])
df2['time'] = pd.to_datetime(df2['time'])

# Sort by time just in case
df1 = df1.sort_values('time')
df2 = df2.sort_values('time')

# Merge on time (keep only overlapping timestamps)
df_merged = pd.merge(df1, df2, on='time', how='inner', suffixes=('_1', '_2'))

# Calculate spearman correlation
# Drop any NaNs
df_clean = df_merged[['flux_1', 'flux_2']].dropna()

corr, p_value = spearmanr(df_clean['flux_1'], df_clean['flux_2'])

# Create the plot
plt.figure(figsize=(12, 6))

plt.text(
    0.76, 0.76,
    f'Spearman Correlation: {corr:.2f}',
    transform=plt.gca().transAxes,
    fontsize=14,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray')
)

plt.plot(df_merged['time'], df_merged['flux_1'],
         label="Grandin's VolcPlume Platform", alpha=0.8, linewidth=1.5)

plt.plot(df_merged['time'], df_merged['flux_2'],
         label='Recreated Code', alpha=0.8, linewidth=1.5)

plt.title('C. SO2 Flux derived from VolcPlume Platform vs Recreated Code: Truncation before Interpolation', fontweight='bold', fontsize=18)
plt.xlabel('Date', fontsize=16)
plt.ylabel('SO2 Flux (kton/day)', fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save the plot
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r'XXX.png', dpi=400, bbox_inches='tight')

In [ ]:
## Appendix Figure A-2 (Part 4): Comparing Flux from VolcPlume Platform and Recreated Code (Truncation after Interpolation)
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# File paths
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-2")  # Navigate to folder with the data files
flux_author = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-08-16_tracks-7_du-0_qa-0_sza-70_P550.csv")
flux_recreated = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-09-01_tracks-7_du-0_qa-0_sza-70_P550.csv") 

# Load the data, skipping comment lines
df1 = pd.read_csv(flux_author, comment='#')
df2 = pd.read_csv(flux_recreated, comment='#')

# Convert time columns to datetime
df1['time'] = pd.to_datetime(df1['time'])
df2['time'] = pd.to_datetime(df2['time'])

# Sort by time just in case
df1 = df1.sort_values('time')
df2 = df2.sort_values('time')

# Merge on time (keep only overlapping timestamps)
df_merged = pd.merge(df1, df2, on='time', how='inner', suffixes=('_1', '_2'))

# Calculate spearman correlation
# Drop any NaNs
df_clean = df_merged[['flux_1', 'flux_2']].dropna()

corr, p_value = spearmanr(df_clean['flux_1'], df_clean['flux_2'])

# Create the plot
plt.figure(figsize=(12, 6))

plt.text(
    0.76, 0.76,
    f'Spearman Correlation: {corr:.2f}',
    transform=plt.gca().transAxes,
    fontsize=14,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray')
)

plt.plot(df_merged['time'], df_merged['flux_1'],
         label="Grandin's VolcPlume Platform", alpha=0.8, linewidth=1.5)

plt.plot(df_merged['time'], df_merged['flux_2'],
         label='Recreated Code', alpha=0.8, linewidth=1.5)

plt.title('D. SO2 Flux derived from VolcPlume Platform vs Recreated Code: Truncation after Interpolation', fontweight='bold', fontsize=18)
plt.xlabel('Date', fontsize=16)
plt.ylabel('SO2 Flux (kton/day)', fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save the plot
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r'XXX.png', dpi=400, bbox_inches='tight')

In [ ]:
## Script to convert ground measured wind data that is used by NOVAC to mimic the format that is used by the SO2 flux calculator
# The pressure level is set to 1000 hPa, an arbitrary valid value to be used within Grandin's (2024) flux calculation tool (see "Appropriate Pressure Value" in the sensitivity_analysis.py script for more details)

# 1. CONFIGURATION
# -------------------------------------------------------------------- INSERT PATH FOR INPUT & OUTPUT FOLDERS HERE ------------------------------------------------------------------
input_file = r"XXX" # Path to the raw ground wind data file
output_dir = r"XXX"

volcano_name = "Merapi"
pressure_level = 1000  # An arbitrary value 
start_date = "2023-06-01"  # Format: YYYY-MM-DD
end_date = "2023-06-04"   # Format: YYYY-MM-DD
target_hour = 12            # 6 means 06:00:00. In this case, we are using the NOVAC measurements taken at 12:00 local time (UTC+7)

# Automatically construct the correct output filename
output_filename = f"ERA5_wind_{volcano_name}_{pressure_level}hPa_{start_date}_{end_date}.csv"
output_file = os.path.join(output_dir, output_filename)

# 2. LOAD AND FORMAT DATA
# Load raw ground data
df_raw = pd.read_csv(input_file, sep='\s+')

# Combine 'date' and 'time' and parse as actual datetime objects for filtering
combined_datetime = df_raw['date'] + ' ' + df_raw['time']
df_raw['datetime'] = pd.to_datetime(combined_datetime, format='%Y.%m.%d %H:%M')


# 3. FILTER DATA (Date Range & 1-per-day)
# Filter to only keep rows within the start and end dates
mask_date = (df_raw['datetime'] >= pd.to_datetime(start_date)) & (df_raw['datetime'] <= pd.to_datetime(end_date) + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))
df_filtered = df_raw[mask_date].copy()

# Filter down to exactly 1 measurement per day matching the target hour
df_daily = df_filtered[df_filtered['datetime'].dt.hour == target_hour].copy()


# 4. CONSTRUCT ERA5-FORMATTED DATAFRAME
df_out = pd.DataFrame()

# The calculator expects standard 'YYYY-MM-DD HH:MM:SS' string format
df_out['Date'] = df_daily['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_out['wind_speed_magnitude_m_per_s'] = df_daily['ws']

# Calculate U (east) and V (north) components
dir_rad = np.deg2rad(df_daily['wd'])
df_out['wind_speed_east_m_per_s'] = -df_daily['ws'] * np.sin(dir_rad)
df_out['wind_speed_north_m_per_s'] = -df_daily['ws'] * np.cos(dir_rad)

df_out['wind_azimuth_clockwise_from_north_deg'] = df_daily['wd']

# ==========================================
# 5. WRITE FILE
# ==========================================
with open(output_file, 'w') as f:
    # Write the mandatory first line
    f.write('# Wind altitude: 0.0 m\n')
    # Append the dataframe (index=False prevents pandas from writing row numbers)
    df_out.to_csv(f, index=False)

print(f"Original rows: {len(df_raw)}")
print(f"Filtered rows (1 per day): {len(df_out)}")
print(f"Successfully generated file: {output_filename}")

In [ ]:
## Appendix Figure A-3: Comparing Estimated Flux Obtained when using ERA5 vs Ground-Measured Wind 
import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# File paths
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-3")  # Navigate to folder with the data files
flux_era5_wind = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-12-31_tracks-10_du-0.2_qa-0_sza-70_drops-none_P550.csv")
flux_ground_wind = os.path.join(data_dir, "Merapi_Flux_7km_2023-06-01_2023-12-31_tracks-10_du-0.2_qa-0_sza-70_drops-none_P1000.csv") 


# Load the data, skipping comment lines
df1 = pd.read_csv(flux_era5_wind, comment='#')
df2 = pd.read_csv(flux_ground_wind, comment='#')

# Convert time columns to datetime
df1['time'] = pd.to_datetime(df1['time'])
df2['time'] = pd.to_datetime(df2['time'])

# Sort by time just in case
df1 = df1.sort_values('time')
df2 = df2.sort_values('time')

# Merge on time (keep only overlapping timestamps)
df_merged = pd.merge(df1, df2, on='time', how='inner', suffixes=('_1', '_2'))

# Calculate spearman correlation
# Drop any NaNs
df_clean = df_merged[['flux_1', 'flux_2']].dropna()

corr, p_value = spearmanr(df_clean['flux_1'], df_clean['flux_2'])

# Create the plot
plt.figure(figsize=(12, 6))

plt.text(
    0.795, 0.85,
    f'Spearman Correlation: {corr:.2f}',
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment='top',
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray')
)

plt.plot(df_merged['time'], df_merged['flux_1'],
         label="ERA5 Wind", alpha=0.8, linewidth=1.5)

plt.plot(df_merged['time'], df_merged['flux_2'],
         label='Ground Measured Wind', alpha=0.8, linewidth=1.5)

plt.title('Comparison of SO2 Flux derived from ERA5 and Ground-Measured Winds \nTracks = 10, DU = 0.2', fontweight='bold', fontsize=15)
plt.xlabel('Date', fontsize=16)
plt.ylabel('SO2 Flux (kton/day)', fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

# Save the plot
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXXX.png")

In [ ]:
## Appendix Figure A-4: Plotting TROPOMI againt Processed Images for 2023
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset
from scipy.spatial import ConvexHull, cKDTree
from scipy.interpolate import griddata
from matplotlib.path import Path
from geopy.distance import distance

# ==========================================
# 1. SETUP & INPUTS
# ==========================================
# 1. Navigate to dataset
script_dir = os.getcwd()  # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-4")  # Navigate to folder with the data files

DAYS_DATA = {
    "22 July 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230722T053338_20230722T071508_29905_03_020500_20230724T095641.nc")
    ],
    "26 July 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230726T055840_20230726T074010_29962_03_020500_20230728T192451.nc")
    ],
    "28 July 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230728T052026_20230728T070155_29990_03_020500_20230730T065725.nc")
    ],
    "1 Sept 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230901T060049_20230901T074219_30487_03_020500_20230903T040627.nc")
    ],
    "12 Sept 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20230912T055330_20230912T073500_30643_03_020500_20230914T032534.nc")
    ],
    "16 Oct 2023": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20231016T051150_20231016T065320_31125_03_020500_20231018T032828.nc")
    ]
}
# Constants
V_LAT, V_LON = -7.54, 110.446
LAT_MIN, LAT_MAX = -13.0, -2.0
LON_MIN, LON_MAX = 105.0, 124.0
VMIN, VMAX = -0.0005, 0.0015 

# Sensitivity Analysis Parameters (For Processed Data)
GRID_RES = 0.05
MAX_GAP_DEG = 1.0
TRACKS_FILTER = 18 
DU_THRESHOLD = 0.5

# Save Path
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
SAVE_DIR = r"XXX"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_sliced_so2_raw(ds, geo, sza_max=70):
    sza = geo['solar_zenith_angle']
    mask = (
        (ds.latitude > LAT_MIN) & (ds.latitude < LAT_MAX) &
        (ds.longitude > LON_MIN) & (ds.longitude < LON_MAX) &
        (ds.qa_value > 0) &
        (sza < sza_max)
    )
    sliced = ds['sulfurdioxide_total_vertical_column'].where(mask, drop=True)
    return sliced if sliced.size > 0 else None

def get_circle_points(lat, lon, radius_km):
    pts = []
    for bearing in range(0, 365, 5):
        dest = distance(kilometers=radius_km).destination((lat, lon), bearing)
        pts.append([dest.longitude, dest.latitude])
    return np.array(pts)

# Calculate radius points once
circle_pts = get_circle_points(V_LAT, V_LON, 500)

# ==========================================
# 3. MAIN PLOTTING LOOP
# ==========================================
n_days = len(DAYS_DATA)

fig, axes = plt.subplots(nrows=n_days, ncols=2, figsize=(18, 5.5 * n_days), 
                         subplot_kw={'projection': ccrs.PlateCarree()})

# Ensure axes is 2D even if there's only 1 day
if n_days == 1:
    axes = np.array([axes])

for i, (date_str, file_paths) in enumerate(DAYS_DATA.items()):
    ax_raw = axes[i, 0]
    ax_proc = axes[i, 1]
    
    print(f"Processing {date_str}...")

    # --- 3a. Setup Map Features for Both Axes ---
    for ax in (ax_raw, ax_proc):
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        
        # Volcano & Radius
        ax.plot(V_LON, V_LAT, 'ro', markersize=8, transform=ccrs.PlateCarree(), label='Mt. Merapi')
        ax.plot(circle_pts[:, 0], circle_pts[:, 1], 'r--', linewidth=2, transform=ccrs.PlateCarree(), label='500km Radius')
        
        gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
        gl.top_labels = False
        gl.right_labels = False

    # ==========================================
    # 3b. LEFT COLUMN: RAW PLOT (xarray)
    # ==========================================
    im_raw = None
    for p_path in file_paths:
        if not os.path.exists(p_path): continue
        ds = xr.open_dataset(p_path, group="PRODUCT").set_coords(['longitude', 'latitude'])
        geo = xr.open_dataset(p_path, group="PRODUCT/SUPPORT_DATA/GEOLOCATIONS")
        
        so2_sliced = get_sliced_so2_raw(ds, geo)
        if so2_sliced is not None:
            # Overlay each file on the same raw axis
            im_raw = so2_sliced.plot(
                ax=ax_raw, x='longitude', y='latitude', transform=ccrs.PlateCarree(),
                vmin=VMIN, vmax=VMAX, cmap='viridis', add_colorbar=False
            )
            
    ax_raw.set_title(f"Raw - {date_str}\n(SZA < 70)", fontsize=16, pad=12)
    ax_raw.legend(loc='upper right')

    # ==========================================
    # 3c. RIGHT COLUMN: PROCESSED PLOT
    # ==========================================
    daily_data_list = []
    for p_path in file_paths:
        if not os.path.exists(p_path): continue
        with Dataset(p_path, 'r') as root:
            p = root.groups['PRODUCT']
            res = p.groups['SUPPORT_DATA'].groups['DETAILED_RESULTS']
            geo = p.groups['SUPPORT_DATA'].groups['GEOLOCATIONS']
            
            v_so2_raw = res.variables['sulfurdioxide_total_vertical_column_7km']
            conv = v_so2_raw.getncattr('multiplication_factor_to_convert_to_DU')
            
            lat_full = p.variables['latitude'][:]
            lon_full = p.variables['longitude'][:]
            
            # Crop
            crop = (lat_full > V_LAT - 6) & (lat_full < V_LAT + 6) & (lon_full > V_LON - 6) & (lon_full < V_LON + 6)
            if not np.any(crop): continue

            # Row indexing
            n_scanlines, n_pixels = p.dimensions['scanline'].size, p.dimensions['ground_pixel'].size
            row_indices = np.tile(np.arange(n_pixels), n_scanlines).reshape(lat_full.shape)

            temp_df = pd.DataFrame({
                'lat': lat_full[crop], 'lon': lon_full[crop],
                'qa': p.variables['qa_value'][:][crop],
                'sza': geo.variables['solar_zenith_angle'][:][crop],
                'row': row_indices[crop],
                'so2': np.ma.filled(v_so2_raw[:].astype(float), np.nan)[crop] * conv
            })
            daily_data_list.append(temp_df)

    im_proc = None
    if daily_data_list:
        master_df = pd.concat(daily_data_list, ignore_index=True)
        
        # Apply Filters
        m_e = (master_df['row'] < TRACKS_FILTER) | (master_df['row'] > (449 - TRACKS_FILTER))
        m_q, m_s = (master_df['qa'] < 0.1), (master_df['sza'] > 70) 
        soundings = master_df[~(m_e | m_q | m_s | master_df['so2'].isna())].copy()

        if not soundings.empty:
            # Interpolation
            lons = np.arange(V_LON-5.5, V_LON+5.5, GRID_RES)
            lats = np.arange(V_LAT-5.5, V_LAT+5.5, GRID_RES)
            grid_lon, grid_lat = np.meshgrid(lons, lats)
            grid_pts = np.column_stack([grid_lon.ravel(), grid_lat.ravel()])

            pts = soundings[['lon', 'lat']].values
            grid_so2 = griddata(pts, soundings['so2'].values, grid_pts, method='linear')

            # Masking
            hull_path = Path(pts[ConvexHull(pts).vertices])
            dists, _ = cKDTree(pts).query(grid_pts)
            mask = hull_path.contains_points(grid_pts) & (dists <= MAX_GAP_DEG)

            # DU Truncation & Conversion to mol/m2
            spatially_masked = np.where(mask, np.nan_to_num(grid_so2, nan=0.0), 0.0)
            grid_so2_final_du = np.where(spatially_masked >= DU_THRESHOLD, spatially_masked, 0.0)
            grid_so2_final_mol_m2 = grid_so2_final_du.reshape(grid_lon.shape) * 4.4615e-4

            # Plot
            im_proc = ax_proc.pcolormesh(
                grid_lon, grid_lat, grid_so2_final_mol_m2, 
                cmap='viridis', vmin=VMIN, vmax=VMAX, transform=ccrs.PlateCarree(), shading='auto'
            )

    ax_proc.set_title(f"Processed - {date_str}\n(Tracks: {TRACKS_FILTER} | DU Thresh: {DU_THRESHOLD})", fontsize=16, pad=12)
    ax_proc.legend(loc='upper right')

# ==========================================
# 4. GLOBAL COLORBAR & FORMATTING
# ==========================================
fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.08, wspace=0.0, hspace=0.25)

# Add the colorbar axes. [left, bottom, width, height]
cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.015]) 

# Use whichever image successfully rendered
if im_raw or im_proc:
    cb = fig.colorbar(im_raw if im_raw else im_proc, cax=cbar_ax, orientation='horizontal', extend='both')
    cb.set_label('SO2 Vertical Column [mol m-2]', fontsize=14)
    # Increase the tick numbers size
    cb.ax.tick_params(labelsize=14)

# Save the master plot
master_filename = os.path.join(SAVE_DIR, f"Comparison_Grid_23.png")
plt.savefig(master_filename, dpi=300, bbox_inches='tight')
print(f"Master plot saved to: {master_filename}")
plt.show()

In [ ]:
## Appendix Figure A-5: Plotting out raw TROPOMI swath on 14th November 2024 to visualise the spike in SO2 mass from the Lewotobi eruption
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import cartopy.crs as ccrs 
import cartopy.feature as cfeature
from geopy.distance import distance

# 1. Navigate to dataset
script_dir = os.getcwd() # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-5")  # Navigate to folder with the data files
path = os.path.join(data_dir, "S5P_OFFL_L2__SO2____20241114T050037_20241114T064207_36729_03_020601_20241117T122455.nc")

# Open PRODUCT group
ds = xr.open_dataset(path, group="PRODUCT").set_coords(['longitude', 'latitude'])

# Open GEOLOCATIONS group
geo = xr.open_dataset(path, group="PRODUCT/SUPPORT_DATA/GEOLOCATIONS")

# 2. Define the geographic slice
lat_min, lat_max = -13.0, -2.0
lon_min, lon_max = 105.0, 124.0

def get_sliced_so2(ds, geo, sza_max=70):
    sza = geo['solar_zenith_angle']

    mask = (
        (ds.latitude > lat_min) & (ds.latitude < lat_max) &
        (ds.longitude > lon_min) & (ds.longitude < lon_max) &
        (ds.qa_value > 0.1) &
        (sza < sza_max)
    )

    sliced = ds['sulfurdioxide_total_vertical_column'].where(mask, drop=True)
    return sliced if sliced.size > 0 else None

so2_sliced = get_sliced_so2(ds, geo)

# 3. Setup Plot
fig = plt.figure(figsize=(12, 10))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)

vmin, vmax = -0.0005, 0.0015 

# 4. Plot single swath
if so2_sliced is not None:
    im = so2_sliced.plot(
        ax=ax,
        x='longitude',
        y='latitude',
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
        cmap='viridis',
        add_colorbar=False
    )

    plt.colorbar(im, ax=ax, label='SO2 Vertical Column [mol m-2]',
                 extend='both', orientation='horizontal', pad=0.08)

# 5. Volcano marker + 500 km radius
v_lat, v_lon = -7.54, 110.446
ax.plot(v_lon, v_lat, 'ro', markersize=8, transform=ccrs.PlateCarree(), label='Mt. Merapi')

def get_circle_points(lat, lon, radius_km):
    pts = []
    for bearing in range(0, 365, 5):
        dest = distance(kilometers=radius_km).destination((lat, lon), bearing)
        pts.append([dest.longitude, dest.latitude])
    return np.array(pts)

circle_pts = get_circle_points(v_lat, v_lon, 500)
ax.plot(circle_pts[:, 0], circle_pts[:, 1], 'r--', linewidth=2,
        transform=ccrs.PlateCarree(), label='500km Radius')

# 6. Formatting
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
plt.title("Raw Image (14 Nov 2024)")
plt.legend(loc='upper right')

gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
plt.savefig(r"XXX.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Appendix Figure A-6: Plotting TROPOMI against Processed Images for 2024-25
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset
from scipy.spatial import ConvexHull, cKDTree
from scipy.interpolate import griddata
from matplotlib.path import Path
from geopy.distance import distance

# ==========================================
# 1. SETUP & INPUTS
# ==========================================
# 1. Navigate to dataset
script_dir = os.getcwd()  # Get the directory of the current script
data_dir = os.path.join(script_dir, "Data Files/Appendix A-6")  # Navigate to folder with the data files

DAYS_DATA = {
    "17 Aug 2024": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20240817T043857_20240817T062027_35466_03_020601_20240819T024431.nc"),
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20240817T062027_20240817T080156_35467_03_020601_20240819T041343.nc")
    ],
    "24 Aug 2024": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20240824T054757_20240824T072927_35566_03_020601_20240826T035854.nc")
    ],
    "28 Apr 2025": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20250428T051832_20250428T070002_39070_03_020701_20250430T025855.nc")
    ],
    "29 Apr 2025": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20250429T045937_20250429T064108_39084_03_020701_20250501T054251.nc")
    ],
    "2 Jun 2025": [
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20250602T042442_20250602T060612_39566_03_020701_20250604T020941.nc"),
        os.path.join(data_dir, "S5P_OFFL_L2__SO2____20250602T060612_20250602T074742_39567_03_020701_20250604T101517.nc")
    ]
}

# Constants
V_LAT, V_LON = -7.54, 110.446
LAT_MIN, LAT_MAX = -13.0, -2.0
LON_MIN, LON_MAX = 105.0, 124.0
VMIN, VMAX = -0.0005, 0.0015 

# Sensitivity Analysis Parameters (For Processed Data)
GRID_RES = 0.05
MAX_GAP_DEG = 1.0
TRACKS_FILTER = 0
DU_THRESHOLD = 0.3

# Save Path
# ---------------------------------------------------------------------- INSERT PATH FOR PLOT FOLDER HERE -----------------------------------------------------------------------
SAVE_DIR = r"XXX"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_sliced_so2_raw(ds, geo, sza_max=70):
    sza = geo['solar_zenith_angle']
    mask = (
        (ds.latitude > LAT_MIN) & (ds.latitude < LAT_MAX) &
        (ds.longitude > LON_MIN) & (ds.longitude < LON_MAX) &
        (ds.qa_value > 0) &
        (sza < sza_max)
    )
    sliced = ds['sulfurdioxide_total_vertical_column'].where(mask, drop=True)
    return sliced if sliced.size > 0 else None

def get_circle_points(lat, lon, radius_km):
    pts = []
    for bearing in range(0, 365, 5):
        dest = distance(kilometers=radius_km).destination((lat, lon), bearing)
        pts.append([dest.longitude, dest.latitude])
    return np.array(pts)

# Calculate radius points once
circle_pts = get_circle_points(V_LAT, V_LON, 500)

# ==========================================
# 3. MAIN PLOTTING LOOP
# ==========================================
n_days = len(DAYS_DATA)

fig, axes = plt.subplots(nrows=n_days, ncols=2, figsize=(18, 5.5 * n_days), 
                         subplot_kw={'projection': ccrs.PlateCarree()})

# Ensure axes is 2D even if there's only 1 day
if n_days == 1:
    axes = np.array([axes])

for i, (date_str, file_paths) in enumerate(DAYS_DATA.items()):
    ax_raw = axes[i, 0]
    ax_proc = axes[i, 1]
    
    print(f"Processing {date_str}...")

    # --- 3a. Setup Map Features for Both Axes ---
    for ax in (ax_raw, ax_proc):
        ax.add_feature(cfeature.COASTLINE, linewidth=1)
        ax.add_feature(cfeature.BORDERS, linestyle=':')
        ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        
        # Volcano & Radius
        ax.plot(V_LON, V_LAT, 'ro', markersize=8, transform=ccrs.PlateCarree(), label='Mt. Merapi')
        ax.plot(circle_pts[:, 0], circle_pts[:, 1], 'r--', linewidth=2, transform=ccrs.PlateCarree(), label='500km Radius')
        
        gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)
        gl.top_labels = False
        gl.right_labels = False

    # ==========================================
    # 3b. LEFT COLUMN: RAW PLOT (xarray)
    # ==========================================
    im_raw = None
    for p_path in file_paths:
        if not os.path.exists(p_path): continue
        ds = xr.open_dataset(p_path, group="PRODUCT").set_coords(['longitude', 'latitude'])
        geo = xr.open_dataset(p_path, group="PRODUCT/SUPPORT_DATA/GEOLOCATIONS")
        
        so2_sliced = get_sliced_so2_raw(ds, geo)
        if so2_sliced is not None:
            # Overlay each file on the same raw axis
            im_raw = so2_sliced.plot(
                ax=ax_raw, x='longitude', y='latitude', transform=ccrs.PlateCarree(),
                vmin=VMIN, vmax=VMAX, cmap='viridis', add_colorbar=False
            )
            
    ax_raw.set_title(f"Raw - {date_str}", fontsize=16, pad=12)
    ax_raw.legend(loc='upper right')

    # ==========================================
    # 3c. RIGHT COLUMN: PROCESSED PLOT
    # ==========================================
    daily_data_list = []
    for p_path in file_paths:
        if not os.path.exists(p_path): continue
        with Dataset(p_path, 'r') as root:
            p = root.groups['PRODUCT']
            res = p.groups['SUPPORT_DATA'].groups['DETAILED_RESULTS']
            geo = p.groups['SUPPORT_DATA'].groups['GEOLOCATIONS']
            
            v_so2_raw = res.variables['sulfurdioxide_total_vertical_column_7km']
            conv = v_so2_raw.getncattr('multiplication_factor_to_convert_to_DU')
            
            lat_full = p.variables['latitude'][:]
            lon_full = p.variables['longitude'][:]
            
            # Crop
            crop = (lat_full > V_LAT - 6) & (lat_full < V_LAT + 6) & (lon_full > V_LON - 6) & (lon_full < V_LON + 6)
            if not np.any(crop): continue

            # Row indexing
            n_scanlines, n_pixels = p.dimensions['scanline'].size, p.dimensions['ground_pixel'].size
            row_indices = np.tile(np.arange(n_pixels), n_scanlines).reshape(lat_full.shape)

            temp_df = pd.DataFrame({
                'lat': lat_full[crop], 'lon': lon_full[crop],
                'qa': p.variables['qa_value'][:][crop],
                'sza': geo.variables['solar_zenith_angle'][:][crop],
                'row': row_indices[crop],
                'so2': np.ma.filled(v_so2_raw[:].astype(float), np.nan)[crop] * conv
            })
            daily_data_list.append(temp_df)

    im_proc = None
    if daily_data_list:
        master_df = pd.concat(daily_data_list, ignore_index=True)
        
        # Apply Filters
        m_e = (master_df['row'] < TRACKS_FILTER) | (master_df['row'] > (449 - TRACKS_FILTER))
        m_q, m_s = (master_df['qa'] < 0.1), (master_df['sza'] > 70) 
        soundings = master_df[~(m_e | m_q | m_s | master_df['so2'].isna())].copy()

        if not soundings.empty:
            # Interpolation
            lons = np.arange(V_LON-5.5, V_LON+5.5, GRID_RES)
            lats = np.arange(V_LAT-5.5, V_LAT+5.5, GRID_RES)
            grid_lon, grid_lat = np.meshgrid(lons, lats)
            grid_pts = np.column_stack([grid_lon.ravel(), grid_lat.ravel()])

            pts = soundings[['lon', 'lat']].values
            grid_so2 = griddata(pts, soundings['so2'].values, grid_pts, method='linear')

            # Masking
            hull_path = Path(pts[ConvexHull(pts).vertices])
            dists, _ = cKDTree(pts).query(grid_pts)
            mask = hull_path.contains_points(grid_pts) & (dists <= MAX_GAP_DEG)

            # DU Truncation & Conversion to mol/m2
            spatially_masked = np.where(mask, np.nan_to_num(grid_so2, nan=0.0), 0.0)
            grid_so2_final_du = np.where(spatially_masked >= DU_THRESHOLD, spatially_masked, 0.0)
            grid_so2_final_mol_m2 = grid_so2_final_du.reshape(grid_lon.shape) * 4.4615e-4

            # Plot
            im_proc = ax_proc.pcolormesh(
                grid_lon, grid_lat, grid_so2_final_mol_m2, 
                cmap='viridis', vmin=VMIN, vmax=VMAX, transform=ccrs.PlateCarree(), shading='auto'
            )

    ax_proc.set_title(f"Processed - {date_str}", fontsize=16, pad=12)
    ax_proc.legend(loc='upper right')

# ==========================================
# 4. GLOBAL COLORBAR & FORMATTING
# ==========================================
fig.subplots_adjust(left=0.05, right=0.95, top=0.95, bottom=0.08, wspace=0.0, hspace=0.25)

# Add the colorbar axes. [left, bottom, width, height]
cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.015]) 

# Use whichever image successfully rendered
if im_raw or im_proc:
    cb = fig.colorbar(im_raw if im_raw else im_proc, cax=cbar_ax, orientation='horizontal', extend='both')
    cb.set_label('SO2 Vertical Column Mass [mol m-2]', fontsize=14)
    # Increase the tick numbers size
    cb.ax.tick_params(labelsize=14)

# Save the master plot
master_filename = os.path.join(SAVE_DIR, f"Comparison_Grid_2425.png")
plt.savefig(master_filename, dpi=300, bbox_inches='tight')
print(f"Master plot saved to: {master_filename}")
plt.show()
